# Orbit Wars JAX V24-Port PPO Train

Notebook nay dung JAX simulator da validate va candidate generator port tu logic V24 o muc thuc dung.
Muc tieu: giu nen heuristic manh hon heuristic JAX don gian, train PPO vectorized de rerank/bias candidate.


In [ ]:
# === JAX TRAIN CONFIG ===
# Cell nay la noi ban sua config chinh. Tat ca comment viet khong dau de tranh loi font tren Kaggle.

from pathlib import Path

# Tensor size co dinh. Neu map lon hon thi tang MAX_PLANETS/MAX_FLEETS, nhung se ton VRAM hon.
MAX_PLANETS = 64
MAX_FLEETS = 256
MAX_MOVES_PER_PLAYER = 8
MAX_PLAYERS = 4
MAX_COMET_GROUPS = 5
MAX_COMET_PATH = 40
COMET_SPAWN_STEPS = (50, 150, 250, 350, 450)

# Ty le train 2P/4P. Muc tieu chinh la 4P, nhung giu 2P de agent khong hoc lech qua FFA.
# Neu chi muon train 4P: TRAIN_PROB_4P = 1.0, TRAIN_PROB_2P = 0.0.
TRAIN_PROB_4P = 1.00
TRAIN_PROB_2P = 0.00

# Quy mo train. Tong samples moi PPO update = NUM_ENVS * ROLLOUT_STEPS.
# Ban toi uu nen chay tren Kaggle GPU/TPU hoac may co JAX GPU. Neu OOM, giam NUM_ENVS truoc.
NUM_ENVS = 512
ROLLOUT_STEPS = 128
TOTAL_PPO_UPDATES = 800

# Reset map sau moi N update de tranh overfit vao mot lo state lien tiep.
RESET_EVERY_UPDATES = 4

# Log/progress. EVAL o day la log reward trong JAX train, benchmark that phai export agent va chay official env.
EVAL_EVERY_UPDATES = 10
SAVE_EVERY_UPDATES = 50  # Luu checkpoint moi 50 update de co nhieu moc test/rollback hon.
EVAL_4P_MATCHES = 64
EVAL_2P_MATCHES = 32

# PPO params.
# GAMMA cao vi Orbit Wars co chien luoc dai; entropy giup policy khong ket cung vao heuristic prior qua som.
GAMMA = 0.995
GAE_LAMBDA = 0.95
PPO_CLIP_EPS = 0.20
ENTROPY_COEF = 0.012
VALUE_COEF = 0.50
LEARNING_RATE = 2.5e-4
UPDATE_EPOCHS = 4
MINIBATCH_SIZE = 4096

# Reward shaping cho PPO.
# Terminal reward van la thuong win/loss cua game. Shaping chi them tin hieu moi step de hoc nhanh hon:
# - production: kha nang snowball
# - planet count: map control
# - ships/fleets: tai san hien co
# - relative lead: vi tri so voi doi thu trong 2P/4P
# Neu thay agent hoc qua phong thu, giam SHAPE_SHIP_WEIGHT/SHAPE_LEAD_WEIGHT.
TERMINAL_REWARD_WEIGHT = 1.00
SHAPING_REWARD_SCALE = 0.030
SHAPE_PROD_WEIGHT = 5.00
SHAPE_PLANET_WEIGHT = 1.20
SHAPE_PLANET_SHIP_WEIGHT = 0.030
SHAPE_FLEET_SHIP_WEIGHT = 0.018
SHAPE_LEAD_WEIGHT = 2.00
SHAPING_REWARD_CLIP = 0.25
TOTAL_REWARD_CLIP = 1.50

# V24-like JAX candidate generator knobs.
# Day la heuristic base de sinh action candidate; PPO hoc chon/rerank candidate, khong hoc action lien tuc tu so 0.
V24_HORIZON = 13
V24_MAX_SOURCES = 6
V24_MAX_ATTACK_TARGETS = 12
V24_MAX_DEF_TARGETS = 4
V24_MAX_WAVES = 6
V24_MAX_STRIKE_SOURCES = 3
V24_MIN_SHIPS_TO_LAUNCH = 4.0
V24_ROI_THRESHOLD = 1.5
V24_SAFE_DRAIN_FRAC = 0.58
V24_SAFE_RESERVE_BASE = 3.0
V24_REGROUP_SEND_FRAC = 0.28

# Policy network. NUM_CANDIDATES phai khop candidate generator.
FEATURE_DIM = 12
NUM_CANDIDATES = 109
HIDDEN_DIM = 96
BIAS_SCALE = 2.0
TRAIN_TEMPERATURE = 1.20

# Notebook run control.
# AUTO_RUN_VALIDATION=True: Run all se check JAX env voi official env truoc.
# AUTO_START_TRAIN=False: Run all khong train dai. Doi True khi da san sang train that.
AUTO_RUN_VALIDATION = True
AUTO_START_TRAIN = False

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT_DIR = WORK_DIR / 'jax_ppo_out'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Resume tu checkpoint policy .npz. Checkpoint cu chi co weights, nen optimizer Adam se khoi tao lai.
# Neu ban co ppo_policy_update_0200.npz thi upload vao Kaggle va set path dung.
RESUME_FROM_POLICY = True
# Neu upload checkpoint 0400 vao Kaggle Dataset/Input, doi duong dan nay toi file .npz do.
# Vi du: Path('/kaggle/input/<ten-dataset>/ppo_policy_update_0400.npz')
RESUME_POLICY_PATH = Path('/kaggle/input/YOUR_CHECKPOINT_DATASET/ppo_policy_update_0400.npz')
RESUME_START_UPDATE = 400  # checkpoint nay da train den update nao.

# JAX simulator validation.
VALIDATE_RANDOM_SEEDS = [1, 2, 3, 4, 5]
VALIDATE_STEPS = 30
VALIDATE_WITH_COMETS = False

# Official opponent paths chi dung cho benchmark/export ve sau, khong dung trong JAX self-play train.
# Train hien tai la PPO self-play tren JAX simulator + V24-like candidate generator.
TRAIN_OPPONENT_POOL_4P = [
    Path('/kaggle/input/YOUR_DATASET_1/agent_a.py'),
    Path('/kaggle/input/YOUR_DATASET_2/agent_b.py'),
    Path('/kaggle/input/YOUR_DATASET_3/agent_c.py'),
    Path('/kaggle/input/YOUR_DATASET_4/agent_d.py'),
    Path('/kaggle/input/YOUR_DATASET_5/agent_e.py'),
    Path('/kaggle/input/YOUR_DATASET_6/agent_f.py'),
    Path('/kaggle/input/YOUR_DATASET_7/agent_g.py'),
]


print('NUM_ENVS', NUM_ENVS)
print('ROLLOUT_STEPS', ROLLOUT_STEPS)
print('TOTAL_PPO_UPDATES', TOTAL_PPO_UPDATES)
print('samples/update', NUM_ENVS * ROLLOUT_STEPS)
print('TRAIN_PROB_4P', TRAIN_PROB_4P, 'TRAIN_PROB_2P', TRAIN_PROB_2P)
print('OUT_DIR', OUT_DIR)


In [ ]:
# === IMPORTS AND OFFICIAL ENGINE ===
# Doc official engine de dung lam ground truth va reset/map generator.

import math
import random
import json
from dataclasses import dataclass
from typing import NamedTuple

import numpy as np
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x
import jax
import jax.numpy as jnp
from jax import lax, random as jrandom

from kaggle_environments import make
from kaggle_environments.envs.orbit_wars import orbit_wars as ow

BOARD_SIZE = float(ow.BOARD_SIZE)
CENTER = float(ow.CENTER)
SUN_RADIUS = float(ow.SUN_RADIUS)
ROTATION_RADIUS_LIMIT = float(ow.ROTATION_RADIUS_LIMIT)
COMET_RADIUS = float(ow.COMET_RADIUS)
COMET_PRODUCTION = float(ow.COMET_PRODUCTION)
COMET_SPAWN_STEPS = tuple(int(x) for x in ow.COMET_SPAWN_STEPS)

print('official orbit_wars loaded')
print('BOARD_SIZE', BOARD_SIZE, 'SUN_RADIUS', SUN_RADIUS, 'ROT_LIMIT', ROTATION_RADIUS_LIMIT)


In [ ]:
# === JAX STATE / ACTION FORMAT ===
# planets: [B, P, 7] = id, owner, x, y, radius, ships, production. Dead slots id=-1.
# fleets:  [B, F, 7] = id, owner, x, y, angle, from_planet_id, ships. Dead slots id=-1.
# actions: [B, A, M, 3] = from_planet_id, angle, ships. Invalid moves ships<=0.
# comet_paths: [B, G, 4, T, 2] precomputed by official generator from hidden seed.

class JaxState(NamedTuple):
    planets: jnp.ndarray
    fleets: jnp.ndarray
    initial_planets: jnp.ndarray
    comet_paths: jnp.ndarray
    comet_lengths: jnp.ndarray
    comet_ships: jnp.ndarray
    comet_base_id: jnp.ndarray
    step: jnp.ndarray
    next_fleet_id: jnp.ndarray
    angular_velocity: jnp.ndarray
    player_count: jnp.ndarray
    done: jnp.ndarray
    rewards: jnp.ndarray

class JaxAction(NamedTuple):
    moves: jnp.ndarray


def empty_state(batch_size: int, player_count: int = 4):
    planets = jnp.full((batch_size, MAX_PLANETS, 7), -1.0, dtype=jnp.float32)
    fleets = jnp.full((batch_size, MAX_FLEETS, 7), -1.0, dtype=jnp.float32)
    return JaxState(
        planets=planets,
        fleets=fleets,
        initial_planets=planets,
        comet_paths=jnp.zeros((batch_size, MAX_COMET_GROUPS, 4, MAX_COMET_PATH, 2), dtype=jnp.float32),
        comet_lengths=jnp.zeros((batch_size, MAX_COMET_GROUPS, 4), dtype=jnp.int32),
        comet_ships=jnp.zeros((batch_size, MAX_COMET_GROUPS), dtype=jnp.float32),
        comet_base_id=jnp.zeros((batch_size,), dtype=jnp.int32),
        step=jnp.zeros((batch_size,), dtype=jnp.int32),
        next_fleet_id=jnp.zeros((batch_size,), dtype=jnp.int32),
        angular_velocity=jnp.zeros((batch_size,), dtype=jnp.float32),
        player_count=jnp.full((batch_size,), int(player_count), dtype=jnp.int32),
        done=jnp.zeros((batch_size,), dtype=jnp.bool_),
        rewards=jnp.zeros((batch_size, MAX_PLAYERS), dtype=jnp.float32),
    )


def pad_rows(rows, max_rows, width=7, fill=-1.0):
    arr = np.full((max_rows, width), fill, dtype=np.float32)
    rows = np.asarray(rows, dtype=np.float32)
    n = min(len(rows), max_rows)
    if n:
        arr[:n] = rows[:n]
    return arr


In [ ]:
# === JAX GEOMETRY MATCHING OFFICIAL ENGINE ===
# Port cac ham geometry tu orbit_wars.py sang JAX.

@jax.jit
def fleet_speed(ships, max_speed=6.0):
    ships = jnp.maximum(ships, 1.0)
    speed = 1.0 + (max_speed - 1.0) * (jnp.log(ships) / jnp.log(1000.0)) ** 1.5
    return jnp.minimum(speed, max_speed)

@jax.jit
def point_to_segment_distance(px, py, vx, vy, wx, wy):
    l2 = (vx - wx) ** 2 + (vy - wy) ** 2
    def nonzero(_):
        t = ((px - vx) * (wx - vx) + (py - vy) * (wy - vy)) / l2
        t = jnp.clip(t, 0.0, 1.0)
        projx = vx + t * (wx - vx)
        projy = vy + t * (wy - vy)
        return jnp.sqrt((px - projx) ** 2 + (py - projy) ** 2)
    def zero(_):
        return jnp.sqrt((px - vx) ** 2 + (py - vy) ** 2)
    return lax.cond(l2 == 0.0, zero, nonzero, operand=None)

@jax.jit
def swept_pair_hit(ax, ay, bx, by, p0x, p0y, p1x, p1y, radius):
    d0x = ax - p0x
    d0y = ay - p0y
    dvx = (bx - ax) - (p1x - p0x)
    dvy = (by - ay) - (p1y - p0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - radius * radius
    disc = b * b - 4.0 * a * c
    def small_a(_):
        return c <= 0.0
    def normal(_):
        sq = jnp.sqrt(jnp.maximum(disc, 0.0))
        t1 = (-b - sq) / (2.0 * a)
        t2 = (-b + sq) / (2.0 * a)
        return (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    return lax.cond(a < 1e-12, small_a, normal, operand=None)


In [ ]:
# === OFFICIAL RESET TO JAX STATE ===
# Dung official engine de tao initial state va pack vao tensor JAX.
# Comet path/ships duoc precompute bang official generator theo hidden seed.

from types import SimpleNamespace


def official_initial_state(num_agents=4, seed=0, episode_steps=500, ship_speed=6.0, comet_speed=4.0):
    state = [
        SimpleNamespace(observation=SimpleNamespace(step=0), action=[], status='ACTIVE', reward=0)
    ] + [
        SimpleNamespace(observation=SimpleNamespace(player=i), action=[], status='ACTIVE', reward=0)
        for i in range(1, num_agents)
    ]
    env = SimpleNamespace(
        configuration=SimpleNamespace(seed=seed, shipSpeed=ship_speed, episodeSteps=episode_steps, cometSpeed=comet_speed),
        done=False,
        info={},
    )
    state = ow.interpreter(state, env)
    # Keep seed only inside this training notebook so schedule can be precomputed.
    state[0].observation.episode_seed = env.info.get('seed', seed)
    return state, env


def precompute_comet_schedule(initial_planets, angular_velocity, episode_seed, comet_speed=4.0):
    paths_arr = np.zeros((MAX_COMET_GROUPS, 4, MAX_COMET_PATH, 2), dtype=np.float32)
    lengths = np.zeros((MAX_COMET_GROUPS, 4), dtype=np.int32)
    ships = np.zeros((MAX_COMET_GROUPS,), dtype=np.float32)
    comet_planet_ids = []
    for gi, spawn_step in enumerate(COMET_SPAWN_STEPS[:MAX_COMET_GROUPS]):
        rng = random.Random(f'orbit_wars-comet-{int(episode_seed)}-{int(spawn_step)}')
        paths = ow.generate_comet_paths(
            initial_planets,
            float(angular_velocity),
            int(spawn_step),
            comet_planet_ids,
            float(comet_speed),
            rng=rng,
        )
        if paths:
            ships[gi] = float(min(rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99)))
            for ci, path in enumerate(paths[:4]):
                n = min(len(path), MAX_COMET_PATH)
                lengths[gi, ci] = n
                if n:
                    paths_arr[gi, ci, :n] = np.asarray(path[:n], dtype=np.float32)
            # Official comet lifetimes are <= 40 and spawn windows are 100 turns apart,
            # so no prior comet remains active at the next spawn in default settings.
            comet_planet_ids = []
    return paths_arr, lengths, ships


def pack_official_states(initial_states, comet_speed=4.0):
    planets = []
    fleets = []
    initials = []
    comet_paths = []
    comet_lengths = []
    comet_ships = []
    comet_base_ids = []
    steps = []
    next_ids = []
    ang = []
    rewards = []
    done = []
    player_counts = []
    for state in initial_states:
        obs = state[0].observation
        planets_np = pad_rows(obs.planets, MAX_PLANETS)
        initials_np = pad_rows(obs.initial_planets, MAX_PLANETS)
        live_initial = [p for p in obs.initial_planets if p[0] >= 0]
        base_id = int(max(p[0] for p in live_initial) + 1) if live_initial else 0
        seed = int(getattr(obs, 'episode_seed', 0))
        paths_np, lengths_np, ships_np = precompute_comet_schedule(
            [list(p) for p in obs.initial_planets],
            float(getattr(obs, 'angular_velocity', 0.0)),
            seed,
            comet_speed=comet_speed,
        )
        planets.append(planets_np)
        fleets.append(pad_rows(getattr(obs, 'fleets', []), MAX_FLEETS))
        initials.append(initials_np)
        comet_paths.append(paths_np)
        comet_lengths.append(lengths_np)
        comet_ships.append(ships_np)
        comet_base_ids.append(base_id)
        steps.append(int(getattr(obs, 'step', 0)))
        next_ids.append(int(getattr(obs, 'next_fleet_id', 0)))
        ang.append(float(getattr(obs, 'angular_velocity', 0.0)))
        rewards.append([float(getattr(s, 'reward', 0.0)) for s in state] + [0.0] * (MAX_PLAYERS - len(state)))
        done.append(all(getattr(s, 'status', 'ACTIVE') == 'DONE' for s in state))
        player_counts.append(len(state))
    return JaxState(
        planets=jnp.asarray(np.stack(planets), dtype=jnp.float32),
        fleets=jnp.asarray(np.stack(fleets), dtype=jnp.float32),
        initial_planets=jnp.asarray(np.stack(initials), dtype=jnp.float32),
        comet_paths=jnp.asarray(np.stack(comet_paths), dtype=jnp.float32),
        comet_lengths=jnp.asarray(np.stack(comet_lengths), dtype=jnp.int32),
        comet_ships=jnp.asarray(np.stack(comet_ships), dtype=jnp.float32),
        comet_base_id=jnp.asarray(comet_base_ids, dtype=jnp.int32),
        step=jnp.asarray(steps, dtype=jnp.int32),
        next_fleet_id=jnp.asarray(next_ids, dtype=jnp.int32),
        angular_velocity=jnp.asarray(ang, dtype=jnp.float32),
        player_count=jnp.asarray(player_counts, dtype=jnp.int32),
        done=jnp.asarray(done, dtype=jnp.bool_),
        rewards=jnp.asarray(np.asarray(rewards, dtype=np.float32)),
    )

# Smoke reset.
states = [official_initial_state(num_agents=4, seed=s)[0] for s in [1, 2]]
jstate = pack_official_states(states)
print(jstate.planets.shape, jstate.fleets.shape, jstate.step, jstate.comet_paths.shape)


In [ ]:
# === JAX STEP CORE ===
# Core JAX simulator ported from official orbit_wars.py.
# Supported: launch, production, orbiting planets, comet schedule, fleet movement/collision, combat, reward.


def _set_row(arr, idx, row):
    return arr.at[idx].set(row)


def _process_one_move(carry, move_data):
    planets, fleets, next_fleet_id, player_id, player_count = carry
    move = move_data
    from_id = move[0].astype(jnp.int32)
    angle = move[1]
    ships = move[2].astype(jnp.int32).astype(jnp.float32)

    p_ids = planets[:, 0].astype(jnp.int32)
    p_alive = p_ids >= 0
    source_mask = p_alive & (p_ids == from_id)
    p_idx = jnp.argmax(source_mask.astype(jnp.int32))
    source_exists = source_mask[p_idx]
    owned = planets[p_idx, 1].astype(jnp.int32) == player_id
    enough = planets[p_idx, 5] >= ships
    valid_player = player_id < player_count
    valid = source_exists & owned & enough & (ships > 0.0) & valid_player

    f_alive = fleets[:, 0] >= 0
    free_mask = ~f_alive
    f_idx = jnp.argmax(free_mask.astype(jnp.int32))
    free_exists = free_mask[f_idx]
    valid = valid & free_exists

    radius = planets[p_idx, 4]
    start_x = planets[p_idx, 2] + jnp.cos(angle) * (radius + 0.1)
    start_y = planets[p_idx, 3] + jnp.sin(angle) * (radius + 0.1)
    new_fleet = jnp.array([
        next_fleet_id.astype(jnp.float32),
        player_id.astype(jnp.float32),
        start_x,
        start_y,
        angle,
        from_id.astype(jnp.float32),
        ships,
    ], dtype=jnp.float32)

    planets2 = lax.cond(
        valid,
        lambda x: x.at[p_idx, 5].add(-ships),
        lambda x: x,
        planets,
    )
    fleets2 = lax.cond(valid, lambda x: x.at[f_idx].set(new_fleet), lambda x: x, fleets)
    next2 = next_fleet_id + valid.astype(jnp.int32)
    return (planets2, fleets2, next2, player_id, player_count), None


def _process_player_moves(carry, player_id):
    planets, fleets, next_fleet_id, actions, player_count = carry
    moves = actions[player_id]
    (planets, fleets, next_fleet_id, _, _), _ = lax.scan(
        _process_one_move,
        (planets, fleets, next_fleet_id, player_id, player_count),
        moves,
    )
    return (planets, fleets, next_fleet_id, actions, player_count), None


def process_launches_single(planets, fleets, next_fleet_id, actions, player_count):
    (planets, fleets, next_fleet_id, _, _), _ = lax.scan(
        _process_player_moves,
        (planets, fleets, next_fleet_id, actions, player_count),
        jnp.arange(MAX_PLAYERS, dtype=jnp.int32),
    )
    return planets, fleets, next_fleet_id


def production_single(planets):
    alive = planets[:, 0] >= 0
    owned = planets[:, 1] != -1
    add = jnp.where(alive & owned, planets[:, 6], 0.0)
    return planets.at[:, 5].add(add)


def _spawn_one_comet_copy(carry, copy_idx):
    planets, initial_planets, group_idx, comet_ships_value, comet_base_id, should_spawn = carry
    free_mask = planets[:, 0] < 0
    p_idx = jnp.argmax(free_mask.astype(jnp.int32))
    free_exists = free_mask[p_idx]
    valid = should_spawn & free_exists
    pid = comet_base_id + copy_idx
    row = jnp.array([
        pid.astype(jnp.float32),
        -1.0,
        -99.0,
        -99.0,
        float(ow.COMET_RADIUS),
        comet_ships_value,
        float(ow.COMET_PRODUCTION),
    ], dtype=jnp.float32)
    planets = lax.cond(valid, lambda x: x.at[p_idx].set(row), lambda x: x, planets)
    initial_planets = lax.cond(valid, lambda x: x.at[p_idx].set(row), lambda x: x, initial_planets)
    return (planets, initial_planets, group_idx, comet_ships_value, comet_base_id, should_spawn), None


def spawn_comets_single(planets, initial_planets, step, comet_lengths, comet_ships, comet_base_id):
    spawn_steps = jnp.asarray(COMET_SPAWN_STEPS, dtype=jnp.int32)
    next_step = step + jnp.asarray(1, dtype=jnp.int32)
    spawn_mask = spawn_steps == next_step
    group_idx = jnp.argmax(spawn_mask.astype(jnp.int32))
    has_spawn_step = jnp.any(spawn_mask)
    has_path = comet_lengths[group_idx, 0] > 0
    should_spawn = has_spawn_step & has_path
    comet_ships_value = comet_ships[group_idx]
    (planets, initial_planets, *_), _ = lax.scan(
        _spawn_one_comet_copy,
        (planets, initial_planets, group_idx, comet_ships_value, comet_base_id, should_spawn),
        jnp.arange(4, dtype=jnp.int32),
    )
    return planets, initial_planets


def compute_planet_paths_single(planets, initial_planets, step, angular_velocity, comet_paths, comet_lengths, comet_base_id):
    old_xy = planets[:, 2:4]
    init_xy = initial_planets[:, 2:4]
    p_ids = planets[:, 0].astype(jnp.int32)
    comet_copy = p_ids - comet_base_id
    is_comet = (planets[:, 0] >= 0) & (comet_copy >= 0) & (comet_copy < 4)

    dx = init_xy[:, 0] - CENTER
    dy = init_xy[:, 1] - CENTER
    orb_r = jnp.sqrt(dx * dx + dy * dy)
    init_angle = jnp.arctan2(dy, dx)
    radius = planets[:, 4]
    alive = planets[:, 0] >= 0
    has_init = initial_planets[:, 0] >= 0
    orbiting = alive & has_init & (~is_comet) & ((orb_r + radius) < ROTATION_RADIUS_LIMIT)
    cur_angle = init_angle + angular_velocity * step.astype(jnp.float32)
    orbit_x = CENTER + orb_r * jnp.cos(cur_angle)
    orbit_y = CENTER + orb_r * jnp.sin(cur_angle)
    orbit_xy = jnp.stack([orbit_x, orbit_y], axis=-1)

    spawn_steps = jnp.asarray(COMET_SPAWN_STEPS, dtype=jnp.int32)
    path_idx_by_group = (step + jnp.asarray(1, dtype=jnp.int32)) - spawn_steps
    active_group_mask = path_idx_by_group >= 0
    group_idx = jnp.argmax(jnp.where(active_group_mask, jnp.arange(MAX_COMET_GROUPS, dtype=jnp.int32), -1))
    path_idx = path_idx_by_group[group_idx]
    safe_copy = jnp.clip(comet_copy, 0, 3)
    safe_path_idx = jnp.clip(path_idx, 0, MAX_COMET_PATH - 1)
    comet_xy_all = comet_paths[group_idx, safe_copy, safe_path_idx]
    comet_len = comet_lengths[group_idx, safe_copy]
    comet_expired = is_comet & (path_idx >= comet_len)
    comet_has_path = is_comet & (comet_len > 0) & (path_idx >= 0)
    comet_new_xy = jnp.where(comet_has_path[:, None] & (~comet_expired)[:, None], comet_xy_all, old_xy)

    new_xy = jnp.where(orbiting[:, None], orbit_xy, old_xy)
    new_xy = jnp.where(is_comet[:, None], comet_new_xy, new_xy)
    comet_first_placement = is_comet & (old_xy[:, 0] < 0.0)
    check = alive & (~comet_first_placement)
    return old_xy, new_xy, check, comet_expired

def _move_one_fleet(carry, f_idx):
    fleets, combat, planets, p_old, p_new, p_check, ship_speed = carry
    fleet = fleets[f_idx]
    alive = fleet[0] >= 0
    owner = fleet[1].astype(jnp.int32)
    angle = fleet[4]
    ships = fleet[6]
    spd = fleet_speed(ships, ship_speed)
    old_x = fleet[2]
    old_y = fleet[3]
    new_x = old_x + jnp.cos(angle) * spd
    new_y = old_y + jnp.sin(angle) * spd

    hit_mask = jax.vmap(lambda ox, oy, nx, ny, r, chk: chk & swept_pair_hit(old_x, old_y, new_x, new_y, ox, oy, nx, ny, r))(
        p_old[:, 0], p_old[:, 1], p_new[:, 0], p_new[:, 1], planets[:, 4], p_check
    )
    hit_any = alive & jnp.any(hit_mask)
    hit_idx = jnp.argmax(hit_mask.astype(jnp.int32))
    in_bounds = (0.0 <= new_x) & (new_x <= BOARD_SIZE) & (0.0 <= new_y) & (new_y <= BOARD_SIZE)
    sun_dist = point_to_segment_distance(CENTER, CENTER, old_x, old_y, new_x, new_y)
    hit_sun = sun_dist < SUN_RADIUS
    remove = hit_any | (alive & (~in_bounds)) | (alive & (~hit_any) & hit_sun)

    fleet_moved = fleet.at[2].set(new_x).at[3].set(new_y)
    dead_fleet = jnp.full_like(fleet, -1.0)
    updated_fleet = jnp.where(remove, dead_fleet, fleet_moved)
    updated_fleet = jnp.where(alive, updated_fleet, fleet)
    fleets = fleets.at[f_idx].set(updated_fleet)

    valid_owner = (owner >= 0) & (owner < MAX_PLAYERS)
    combat = lax.cond(
        hit_any & valid_owner,
        lambda c: c.at[hit_idx, owner].add(ships),
        lambda c: c,
        combat,
    )
    return (fleets, combat, planets, p_old, p_new, p_check, ship_speed), None


def move_fleets_single(fleets, planets, p_old, p_new, p_check, ship_speed=6.0):
    combat = jnp.zeros((MAX_PLANETS, MAX_PLAYERS), dtype=jnp.float32)
    (fleets, combat, *_), _ = lax.scan(
        _move_one_fleet,
        (fleets, combat, planets, p_old, p_new, p_check, jnp.asarray(ship_speed, dtype=jnp.float32)),
        jnp.arange(MAX_FLEETS, dtype=jnp.int32),
    )
    return fleets, combat


def apply_planet_movement_single(planets, p_new):
    return planets.at[:, 2:4].set(p_new)


def _combat_one_planet(planets, args):
    p_idx, ships_by_owner = args
    planet = planets[p_idx]
    alive = planet[0] >= 0
    total_attack = ships_by_owner.sum()
    top_owner = jnp.argmax(ships_by_owner).astype(jnp.int32)
    top_ships = ships_by_owner[top_owner]
    top_count = jnp.sum((ships_by_owner == top_ships) & (ships_by_owner > 0.0))
    second_ships = jnp.max(jnp.where(jnp.arange(MAX_PLAYERS) == top_owner, -1.0, ships_by_owner))
    survivor = jnp.where(top_count > 1, 0.0, top_ships - jnp.maximum(second_ships, 0.0))
    has_survivor = alive & (total_attack > 0.0) & (survivor > 0.0)
    same_owner = planet[1].astype(jnp.int32) == top_owner
    reinforced = planet.at[5].add(survivor)
    attacked_ships = planet[5] - survivor
    captured = attacked_ships < 0.0
    attacked = planet.at[5].set(jnp.where(captured, -attacked_ships, attacked_ships)).at[1].set(jnp.where(captured, top_owner.astype(jnp.float32), planet[1]))
    new_planet = lax.cond(same_owner, lambda _: reinforced, lambda _: attacked, operand=None)
    planets = planets.at[p_idx].set(jnp.where(has_survivor, new_planet, planet))
    return planets, None


def combat_single(planets, combat):
    args = (jnp.arange(MAX_PLANETS, dtype=jnp.int32), combat)
    planets, _ = lax.scan(_combat_one_planet, planets, args)
    return planets


def termination_reward_single(planets, fleets, step, player_count, episode_steps=500):
    p_alive = planets[:, 0] >= 0
    f_alive = fleets[:, 0] >= 0
    p_owner = planets[:, 1].astype(jnp.int32)
    f_owner = fleets[:, 1].astype(jnp.int32)
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)
    active_by_planet = jax.vmap(lambda pl: jnp.any(p_alive & (p_owner == pl)))(players)
    active_by_fleet = jax.vmap(lambda pl: jnp.any(f_alive & (f_owner == pl)))(players)
    valid_player = players < player_count
    alive_players = (active_by_planet | active_by_fleet) & valid_player
    terminated = (step >= (episode_steps - 2)) | (jnp.sum(alive_players.astype(jnp.int32)) <= 1)

    planet_scores = jax.vmap(lambda pl: jnp.sum(jnp.where(p_alive & (p_owner == pl), planets[:, 5], 0.0)))(players)
    fleet_scores = jax.vmap(lambda pl: jnp.sum(jnp.where(f_alive & (f_owner == pl), fleets[:, 6], 0.0)))(players)
    scores = planet_scores + fleet_scores
    max_score = jnp.max(jnp.where(valid_player, scores, -1.0))
    rewards = jnp.where((scores == max_score) & (max_score > 0.0) & valid_player, 1.0, -1.0)
    rewards = jnp.where(terminated, rewards, jnp.zeros_like(rewards))
    return terminated, rewards


def remove_expired_comets_single(planets, initial_planets, comet_expired):
    dead_planet = jnp.full((7,), -1.0, dtype=jnp.float32)
    planets = jnp.where(comet_expired[:, None], dead_planet[None, :], planets)
    initial_planets = jnp.where(comet_expired[:, None], dead_planet[None, :], initial_planets)
    return planets, initial_planets


def jax_step_single(state: JaxState, action: JaxAction, ship_speed=6.0, episode_steps=500):
    planets, initial_planets = spawn_comets_single(
        state.planets,
        state.initial_planets,
        state.step,
        state.comet_lengths,
        state.comet_ships,
        state.comet_base_id,
    )
    planets, fleets, next_id = process_launches_single(planets, state.fleets, state.next_fleet_id, action.moves, state.player_count)
    planets = production_single(planets)
    p_old, p_new, p_check, comet_expired = compute_planet_paths_single(
        planets,
        initial_planets,
        state.step,
        state.angular_velocity,
        state.comet_paths,
        state.comet_lengths,
        state.comet_base_id,
    )
    fleets, combat = move_fleets_single(fleets, planets, p_old, p_new, p_check, ship_speed=ship_speed)
    planets = apply_planet_movement_single(planets, p_new)
    planets, initial_planets = remove_expired_comets_single(planets, initial_planets, comet_expired)
    planets = combat_single(planets, combat)
    done, rewards = termination_reward_single(planets, fleets, state.step, state.player_count, episode_steps=episode_steps)
    return JaxState(
        planets=planets,
        fleets=fleets,
        initial_planets=initial_planets,
        comet_paths=state.comet_paths,
        comet_lengths=state.comet_lengths,
        comet_ships=state.comet_ships,
        comet_base_id=state.comet_base_id,
        step=state.step + jnp.asarray(1, dtype=jnp.int32),
        next_fleet_id=next_id,
        angular_velocity=state.angular_velocity,
        player_count=state.player_count,
        done=done,
        rewards=rewards,
    )


jax_step = jax.jit(jax.vmap(jax_step_single, in_axes=(0, 0, None, None)))
print('jax_step core with comet schedule ready')


In [ ]:
# === VALIDATION HARNESS DESIGN ===
# So sanh official engine va JAX engine tren cung initial state/action sequence.
# Khi port xong jax_step, chay ham nay. Neu fail thi khong train PPO.


def compare_planets_np(a, b, atol=1e-5):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.shape != b.shape:
        return False, f'shape mismatch {a.shape} vs {b.shape}'
    diff = np.nanmax(np.abs(a - b)) if a.size else 0.0
    return diff <= atol, f'max_abs_diff={diff}'


def validate_jax_step_against_official(jax_step_fn, seeds=VALIDATE_RANDOM_SEEDS, steps=VALIDATE_STEPS, num_agents=4):
    # Action empty de validate production/rotation/movement no-action.
    # Sau do them scripted launches de validate launch/combat.
    for seed in seeds:
        official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
        js = pack_official_states([official_state])
        for t in range(steps):
            for s in official_state:
                s.action = []
            official_state = ow.interpreter(official_state, official_env)
            # Kaggle wrapper thuong tang step ben ngoai interpreter; test local can tang tay.
            next_step = getattr(official_state[0].observation, 'step', 0) + 1
            for s in official_state:
                s.observation.step = next_step
            empty_actions = JaxAction(moves=jnp.zeros((1, MAX_PLAYERS, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32))
            js = jax_step_fn(js, empty_actions, 6.0, 500)
            jp = np.asarray(js.planets[0])
            op = pad_rows(official_state[0].observation.planets, MAX_PLANETS)
            ok, msg = compare_planets_np(jp, op, atol=1e-4)
            if not ok:
                print('FAIL seed', seed, 't', t, msg)
                return False
    print('validation passed')
    return True

# Chay sau cell jax_step:
# validate_jax_step_against_official(jax_step, seeds=[1], steps=10, num_agents=4)


def pack_actions_for_jax(action_lists, batch_size=1):
    arr = np.zeros((batch_size, MAX_PLAYERS, MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    for b, per_player in enumerate(action_lists):
        for player_id, moves in enumerate(per_player):
            for m, move in enumerate(moves[:MAX_MOVES_PER_PLAYER]):
                arr[b, player_id, :] = arr[b, player_id, :]  # keep shape materialized
                arr[b, player_id, m] = np.asarray(move, dtype=np.float32)
    return JaxAction(moves=jnp.asarray(arr, dtype=jnp.float32))


def compare_fleets_np(a, b, atol=1e-5):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    if a.shape != b.shape:
        return False, f'shape mismatch {a.shape} vs {b.shape}'
    diff = np.nanmax(np.abs(a - b)) if a.size else 0.0
    return diff <= atol, f'max_abs_diff={diff}'


def validate_scripted_launch_once(jax_step_fn, seed=1, num_agents=4):
    official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
    js = pack_official_states([official_state])
    obs = official_state[0].observation
    home = next(p for p in obs.planets if int(p[1]) == 0)
    # Use a tiny launch from player 0. This validates action processing, source debit,
    # production, fleet spawn, fleet movement, and planet rotation for one tick.
    actions = [[[int(home[0]), 0.0, 1]]] + [[] for _ in range(num_agents - 1)]
    for i, st in enumerate(official_state):
        st.action = actions[i]
    official_state = ow.interpreter(official_state, official_env)
    next_step = getattr(official_state[0].observation, 'step', 0) + 1
    for st in official_state:
        st.observation.step = next_step
    ja = pack_actions_for_jax([actions])
    js = jax_step_fn(js, ja, 6.0, 500)
    okp, msgp = compare_planets_np(np.asarray(js.planets[0]), pad_rows(official_state[0].observation.planets, MAX_PLANETS), atol=1e-4)
    okf, msgf = compare_fleets_np(np.asarray(js.fleets[0]), pad_rows(official_state[0].observation.fleets, MAX_FLEETS), atol=1e-4)
    if not okp:
        print('PLANET FAIL', msgp)
    if not okf:
        print('FLEET FAIL', msgf)
    if okp and okf:
        print('scripted launch validation passed')
    return bool(okp and okf)

# Chay sau cell jax_step:
# validate_scripted_launch_once(jax_step, seed=1, num_agents=4)


def make_official_state(planets, fleets, num_agents=2, step=1, angular_velocity=0.01, next_fleet_id=100):
    base = SimpleNamespace(
        observation=SimpleNamespace(
            step=step,
            planets=[list(x) for x in planets],
            fleets=[list(x) for x in fleets],
            next_fleet_id=next_fleet_id,
            angular_velocity=angular_velocity,
            initial_planets=[list(x) for x in planets],
            comets=[],
            comet_planet_ids=[],
        ),
        action=[],
        status='ACTIVE',
        reward=0,
    )
    return [base] + [SimpleNamespace(observation=SimpleNamespace(player=i), action=[], status='ACTIVE', reward=0) for i in range(1, num_agents)]


def run_case_compare(jax_step_fn, name, planets, fleets, num_agents=2, step=1, angular_velocity=0.01, ship_speed=6.0, episode_steps=500):
    official_state = make_official_state(planets, fleets, num_agents=num_agents, step=step, angular_velocity=angular_velocity)
    official_env = SimpleNamespace(configuration=SimpleNamespace(shipSpeed=ship_speed, episodeSteps=episode_steps, cometSpeed=4), done=False)
    js = pack_official_states([official_state])
    official_state = ow.interpreter(official_state, official_env)
    next_step = getattr(official_state[0].observation, 'step', 0) + 1
    for st in official_state:
        st.observation.step = next_step
    empty_actions = JaxAction(moves=jnp.zeros((1, MAX_PLAYERS, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32))
    js = jax_step_fn(js, empty_actions, float(ship_speed), int(episode_steps))
    okp, msgp = compare_planets_np(np.asarray(js.planets[0]), pad_rows(official_state[0].observation.planets, MAX_PLANETS), atol=1e-4)
    okf, msgf = compare_fleets_np(np.asarray(js.fleets[0]), pad_rows(official_state[0].observation.fleets, MAX_FLEETS), atol=1e-4)
    jr = np.asarray(js.rewards[0])[:num_agents]
    orr = np.asarray([float(s.reward) for s in official_state], dtype=np.float32)
    okr = np.max(np.abs(jr - orr)) <= 1e-5
    if not (okp and okf and okr):
        print('FAIL', name, 'planets', msgp, 'fleets', msgf, 'jax_rewards', jr, 'official_rewards', orr)
        return False
    return True


def validate_hard_cases(jax_step_fn):
    cases = [
        ('fleet_hits_sun', [[0,0,80,50,3,50,1]], [[0,0,60,50,math.pi,0,10]], 2, 1, 0.01, 6.0, 500),
        ('fleet_leaves_board', [[0,0,80,50,3,50,1]], [[0,0,99.5,50,0.0,0,10]], 2, 1, 0.01, 6.0, 500),
        ('fleet_survives', [[0,0,80,80,3,50,1]], [[0,0,50,30,0.0,0,10]], 2, 1, 0.01, 6.0, 500),
        ('fast_hit_before_board', [[0,1,98.0,50.0,2.0,50,1]], [[0,0,95.0,50.0,0.0,99,1000]], 2, 1, 0.01, 6.0, 500),
        ('fast_hit_before_sun', [[0,1,62.0,50.0,2.0,50,1]], [[0,0,65.0,50.0,math.pi,99,1000]], 2, 1, 0.0, 6.0, 500),
        ('combat_capture', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,30]], 2, 1, 0.01, 6.0, 500),
        ('combat_reinforce', [[0,0,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,25]], 2, 1, 0.01, 6.0, 500),
        ('combat_insufficient', [[0,-1,80,50,3,20,1]], [[0,0,76.0,50.0,0.0,99,5]], 2, 1, 0.01, 6.0, 500),
        ('two_attackers_capture', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,50],[1,1,76.0,50.0,0.0,99,30]], 2, 1, 0.01, 6.0, 500),
        ('two_attackers_tie', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,30],[1,1,76.0,50.0,0.0,99,30]], 2, 1, 0.01, 6.0, 500),
        ('winner_reinforces', [[0,0,80,50,3,15,1]], [[0,0,76.0,50.0,0.0,99,40],[1,1,76.0,50.0,0.0,99,25]], 2, 1, 0.01, 6.0, 500),
        ('winner_attacks_enemy', [[0,1,80,50,3,5,1]], [[0,0,76.0,50.0,0.0,99,50],[1,2,76.0,50.0,0.0,99,20]], 4, 1, 0.01, 6.0, 500),
        ('same_owner_sum', [[0,-1,80,50,3,10,1]], [[0,0,76.0,50.0,0.0,99,20],[1,0,76.0,50.0,0.0,99,15]], 2, 1, 0.01, 6.0, 500),
        ('reward_max_steps', [[0,0,80,80,3,50,1],[1,1,20,20,3,30,1]], [], 2, 498, 0.01, 6.0, 500),
        ('reward_tie', [[0,0,80,80,3,30,1],[1,1,20,20,3,30,1]], [], 2, 498, 0.01, 6.0, 500),
        ('reward_all_elim', [[0,-1,80,80,3,50,1]], [], 2, 1, 0.01, 6.0, 500),
        ('reward_4p_elim', [[0,2,80,80,3,40,1]], [], 4, 1, 0.01, 6.0, 500),
        ('reward_includes_fleet', [[0,0,80,80,3,10,1],[1,1,20,20,3,30,1]], [[0,0,50,30,0.0,0,50]], 2, 498, 0.01, 6.0, 500),
        ('rotating_swept_collision', [[0,-1,50.0,52.0,1.0,10,0]], [[0,0,49.0,50.0,0.0,1,1000]], 2, 1, math.pi, 2.0, 500),
    ]
    ok_all = True
    for case in cases:
        ok = run_case_compare(jax_step_fn, *case)
        print(case[0], 'OK' if ok else 'FAIL')
        ok_all = ok_all and ok
    print('hard cases', 'PASSED' if ok_all else 'FAILED')
    return ok_all

# Chay sau cell jax_step:
# validate_hard_cases(jax_step)



def canonical_live_rows(arr):
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim != 2 or arr.shape[0] == 0:
        return arr
    live = arr[arr[:, 0] >= 0]
    if live.shape[0] == 0:
        return live
    order = np.argsort(live[:, 0], kind='stable')
    return live[order]


def compare_live_rows_np(a, b, name='rows', atol=1e-5):
    aa = canonical_live_rows(a)
    bb = canonical_live_rows(b)
    if aa.shape != bb.shape:
        return False, f'{name} live shape mismatch {aa.shape} vs {bb.shape}'
    diff = np.nanmax(np.abs(aa - bb)) if aa.size else 0.0
    return diff <= atol, f'{name} max_abs_diff={diff}'


def scripted_multi_step_actions(official_state, seed, turn, num_agents):
    # Deterministic pseudo-random legal-ish launches. This is only for simulator validation.
    rng = random.Random((int(seed) + 17) * 1000003 + int(turn) * 97 + int(num_agents))
    obs = official_state[0].observation
    actions = [[] for _ in range(num_agents)]
    for player_id in range(num_agents):
        owned = [p for p in obs.planets if int(p[1]) == player_id and int(p[5]) >= 4]
        owned.sort(key=lambda p: (-int(p[5]), -int(p[6]), int(p[0])))
        for p in owned[:2]:
            if rng.random() > 0.55:
                continue
            ships = max(1, min(int(p[5]) // 5, 8))
            angle = rng.random() * 2.0 * math.pi
            actions[player_id].append([int(p[0]), float(angle), int(ships)])
    return actions


def compare_step_state(js, official_state, num_agents, atol=1e-4):
    jp = np.asarray(js.planets[0])
    jf = np.asarray(js.fleets[0])
    op = pad_rows(official_state[0].observation.planets, MAX_PLANETS)
    of = pad_rows(official_state[0].observation.fleets, MAX_FLEETS)
    okp, msgp = compare_live_rows_np(jp, op, name='planets', atol=atol)
    okf, msgf = compare_live_rows_np(jf, of, name='fleets', atol=atol)
    jr = np.asarray(js.rewards[0])[:num_agents]
    orr = np.asarray([float(s.reward) for s in official_state], dtype=np.float32)
    okr = np.max(np.abs(jr - orr)) <= 1e-5
    return okp and okf and okr, (msgp, msgf, f'rewards jax={jr} official={orr}')


def validate_random_scripted_no_comet(jax_step_fn, seeds=(1, 2, 3), steps=40, num_agents=4):
    # Official first comet spawns at step 50, so keep this below that until comet support is ported.
    # This validates multi-turn launch, production, rotation, fleet movement, collisions, and rewards.
    if steps >= 49:
        raise ValueError('Use steps < 49 before comet support is implemented.')
    for seed in seeds:
        official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
        js = pack_official_states([official_state])
        ok, msg = compare_step_state(js, official_state, num_agents)
        if not ok:
            print('INITIAL FAIL seed', seed, msg)
            return False
        for turn in range(steps):
            actions = scripted_multi_step_actions(official_state, seed, turn, num_agents)
            for i, st in enumerate(official_state):
                st.action = actions[i]
            official_state = ow.interpreter(official_state, official_env)
            next_step = getattr(official_state[0].observation, 'step', 0) + 1
            for st in official_state:
                st.observation.step = next_step
            js = jax_step_fn(js, pack_actions_for_jax([actions]), 6.0, 500)
            ok, msg = compare_step_state(js, official_state, num_agents, atol=2e-4)
            if not ok:
                print('RANDOM SCRIPTED FAIL seed', seed, 'turn', turn, 'actions', actions)
                print(msg)
                return False
    print('random scripted no-comet validation passed')
    return True

# Chay sau cell jax_step:
# validate_random_scripted_no_comet(jax_step, seeds=(1, 2, 3), steps=40, num_agents=4)



def validate_random_scripted_with_comets(jax_step_fn, seeds=(1, 2, 3), steps=80, num_agents=4):
    # Same scripted actions as the no-comet test, but runs across the first comet spawn.
    for seed in seeds:
        official_state, official_env = official_initial_state(num_agents=num_agents, seed=seed)
        js = pack_official_states([official_state])
        for turn in range(steps):
            actions = scripted_multi_step_actions(official_state, seed, turn, num_agents)
            for i, st in enumerate(official_state):
                st.action = actions[i]
            official_state = ow.interpreter(official_state, official_env)
            next_step = getattr(official_state[0].observation, 'step', 0) + 1
            for st in official_state:
                st.observation.step = next_step
            js = jax_step_fn(js, pack_actions_for_jax([actions]), 6.0, 500)
            ok, msg = compare_step_state(js, official_state, num_agents, atol=3e-4)
            if not ok:
                print('RANDOM COMET FAIL seed', seed, 'turn', turn, 'official_step', official_state[0].observation.step)
                print('actions', actions)
                print(msg)
                return False
    print('random scripted with-comets validation passed')
    return True

# Chay sau cell jax_step:
# validate_random_scripted_with_comets(jax_step, seeds=(1, 2, 3), steps=80, num_agents=4)


In [ ]:
# === AUTO VALIDATION RUN ===
# Cell nay tu chay khi Run all neu AUTO_RUN_VALIDATION=True.
# Neu co fail, notebook raise error de dung truoc khi train.


def run_all_validations():
    print('=== Orbit Wars JAX validation start ===')
    results = []

    print('[1/5] no-action + comet schedule')
    results.append(('noaction_60', validate_jax_step_against_official(jax_step, seeds=[1, 2, 3], steps=60, num_agents=4)))

    print('[2/5] scripted launch')
    results.append(('scripted_launch', validate_scripted_launch_once(jax_step, seed=1, num_agents=4)))

    print('[3/5] hard cases')
    results.append(('hard_cases', validate_hard_cases(jax_step)))

    print('[4/5] random scripted before comet')
    results.append(('random_no_comet_40', validate_random_scripted_no_comet(jax_step, seeds=(1, 2, 3), steps=40, num_agents=4)))

    print('[5/5] random scripted with comet')
    results.append(('random_with_comets_80', validate_random_scripted_with_comets(jax_step, seeds=(1, 2, 3), steps=80, num_agents=4)))

    print('=== Validation summary ===')
    ok_all = True
    for name, ok in results:
        print(f'{name}: {"PASS" if ok else "FAIL"}')
        ok_all = ok_all and bool(ok)
    print('OVERALL:', 'PASS' if ok_all else 'FAIL')
    if not ok_all:
        raise RuntimeError('JAX simulator validation failed. Do not train before fixing this.')
    return True


VALIDATION_OK = False
if AUTO_RUN_VALIDATION:
    VALIDATION_OK = run_all_validations()
else:
    print('AUTO_RUN_VALIDATION=False, skipped validation.')


In [ ]:
# === JAX PPO TRAINING CORE - V24-LIKE CANDIDATE PORT ===
# This is not the old tiny heuristic. It ports the main V24 planning ideas into JAX:
# source shortlist, attack/defense shortlist, safe drain, single waves, focus-fire,
# regroup-style reinforcement, and learned candidate reranking.

class PolicyParams(NamedTuple):
    w1: jnp.ndarray
    b1: jnp.ndarray
    w2: jnp.ndarray
    b2: jnp.ndarray
    value_w1: jnp.ndarray
    value_b1: jnp.ndarray
    value_w2: jnp.ndarray
    value_b2: jnp.ndarray

class AdamState(NamedTuple):
    m: PolicyParams
    v: PolicyParams
    t: jnp.ndarray


def tree_zeros_like(tree):
    return jax.tree_util.tree_map(jnp.zeros_like, tree)


def init_policy_params(key):
    k1, k2, k3, k4 = jrandom.split(key, 4)
    return PolicyParams(
        w1=jrandom.normal(k1, (FEATURE_DIM, HIDDEN_DIM)) * 0.02,
        b1=jnp.zeros((HIDDEN_DIM,)),
        w2=jrandom.normal(k2, (HIDDEN_DIM, 1)) * 0.02,
        b2=jnp.zeros((1,)),
        value_w1=jrandom.normal(k3, (FEATURE_DIM, HIDDEN_DIM)) * 0.02,
        value_b1=jnp.zeros((HIDDEN_DIM,)),
        value_w2=jrandom.normal(k4, (HIDDEN_DIM, 1)) * 0.02,
        value_b2=jnp.zeros((1,)),
    )


def init_adam(params):
    return AdamState(m=tree_zeros_like(params), v=tree_zeros_like(params), t=jnp.asarray(0, dtype=jnp.int32))


def adam_update(params, grads, opt_state, lr=LEARNING_RATE, beta1=0.9, beta2=0.999, eps=1e-8):
    t = opt_state.t + 1
    m = jax.tree_util.tree_map(lambda m, g: beta1 * m + (1.0 - beta1) * g, opt_state.m, grads)
    v = jax.tree_util.tree_map(lambda v, g: beta2 * v + (1.0 - beta2) * (g * g), opt_state.v, grads)
    mhat = jax.tree_util.tree_map(lambda x: x / (1.0 - beta1 ** t), m)
    vhat = jax.tree_util.tree_map(lambda x: x / (1.0 - beta2 ** t), v)
    params = jax.tree_util.tree_map(lambda p, mh, vh: p - lr * mh / (jnp.sqrt(vh) + eps), params, mhat, vhat)
    return params, AdamState(m=m, v=v, t=t)


def make_initial_jax_batch(seed_start=0, batch_size=NUM_ENVS, prob_4p=TRAIN_PROB_4P):
    states = []
    rng = random.Random(int(seed_start))
    for i in range(batch_size):
        num_agents = 4 if rng.random() < prob_4p else 2
        seed = int(seed_start) * 100000 + i + 1
        states.append(official_initial_state(num_agents=num_agents, seed=seed)[0])
    return pack_official_states(states)


def _topk_masked(values, mask, k):
    ranked = jnp.where(mask, values, -1.0e9)
    vals, idx = lax.top_k(ranked, k)
    exists = vals > -1.0e8
    return idx.astype(jnp.int32), exists, vals


def _angle_between(sx, sy, tx, ty):
    return jnp.arctan2(ty - sy, tx - sx)


def _future_planet_xy(state: JaxState, idx, eta):
    planet = state.planets[idx]
    init = state.initial_planets[idx]
    dx = init[..., 2] - CENTER
    dy = init[..., 3] - CENTER
    orb_r = jnp.sqrt(dx * dx + dy * dy)
    init_angle = jnp.arctan2(dy, dx)
    future_step = state.step.astype(jnp.float32) + jnp.ceil(jnp.maximum(eta, 0.0))
    orbiting = (planet[..., 0] >= 0) & (init[..., 0] >= 0) & ((orb_r + planet[..., 4]) < ROTATION_RADIUS_LIMIT)
    is_comet = (planet[..., 0] >= state.comet_base_id) & (planet[..., 0] < state.comet_base_id + 4)
    cur_angle = init_angle + state.angular_velocity * future_step
    fx = CENTER + orb_r * jnp.cos(cur_angle)
    fy = CENTER + orb_r * jnp.sin(cur_angle)
    fx = jnp.where(orbiting & (~is_comet), fx, planet[..., 2])
    fy = jnp.where(orbiting & (~is_comet), fy, planet[..., 3])
    return fx, fy


def _sun_line_clear(ax, ay, bx, by):
    abx = bx - ax
    aby = by - ay
    denom = jnp.maximum(abx * abx + aby * aby, 1e-9)
    u = jnp.clip(((CENTER - ax) * abx + (CENTER - ay) * aby) / denom, 0.0, 1.0)
    cx = ax + u * abx
    cy = ay + u * aby
    d = jnp.sqrt((cx - CENTER) ** 2 + (cy - CENTER) ** 2)
    return d >= (SUN_RADIUS + 0.15)


def _projected_capture_need(target_owner, target_ships, target_prod, eta):
    grows = target_owner >= 0
    turns = jnp.ceil(jnp.maximum(eta, 0.0))
    projected = target_ships + jnp.where(grows, target_prod * turns, 0.0)
    return projected + 1.0


def _planet_xy_at_step(state: JaxState, step_offset):
    planets = state.planets
    init = state.initial_planets
    ids = planets[:, 0].astype(jnp.int32)
    dx = init[:, 2] - CENTER
    dy = init[:, 3] - CENTER
    orb_r = jnp.sqrt(dx * dx + dy * dy)
    init_angle = jnp.arctan2(dy, dx)
    future_step = state.step.astype(jnp.float32) + step_offset.astype(jnp.float32)
    orbiting = (planets[:, 0] >= 0) & (init[:, 0] >= 0) & ((orb_r + planets[:, 4]) < ROTATION_RADIUS_LIMIT)
    is_comet = (ids >= state.comet_base_id) & (ids < state.comet_base_id + 4)
    cur_angle = init_angle + state.angular_velocity * future_step
    px = CENTER + orb_r * jnp.cos(cur_angle)
    py = CENTER + orb_r * jnp.sin(cur_angle)
    px = jnp.where(orbiting & (~is_comet), px, planets[:, 2])
    py = jnp.where(orbiting & (~is_comet), py, planets[:, 3])
    return px, py


def _fleet_first_contact_table(state: JaxState):
    # V24-like target inference for observed fleets. This is scoring-only; the validated
    # simulator remains the source of truth for actual movement/combat.
    planets = state.planets
    p_alive = planets[:, 0] >= 0
    p_ids = planets[:, 0].astype(jnp.int32)
    p_is_comet = (p_ids >= state.comet_base_id) & (p_ids < state.comet_base_id + 4)
    p_check = p_alive & (~p_is_comet)
    steps = jnp.arange(1, V24_HORIZON + 1, dtype=jnp.int32)

    def infer_one(fleet):
        alive = fleet[0] >= 0
        angle = fleet[4]
        ships = jnp.maximum(fleet[6], 1.0)
        spd = fleet_speed(ships, 6.0)
        fx = fleet[2]
        fy = fleet[3]
        ca = jnp.cos(angle)
        sa = jnp.sin(angle)

        def step_hit(k):
            kf = k.astype(jnp.float32)
            old_x = fx + ca * spd * (kf - 1.0)
            old_y = fy + sa * spd * (kf - 1.0)
            new_x = fx + ca * spd * kf
            new_y = fy + sa * spd * kf
            px_old, py_old = _planet_xy_at_step(state, kf - 1.0)
            px_new, py_new = _planet_xy_at_step(state, kf)
            hit_mask = jax.vmap(lambda ox, oy, nx, ny, r, chk: chk & swept_pair_hit(old_x, old_y, new_x, new_y, ox, oy, nx, ny, r))(
                px_old, py_old, px_new, py_new, planets[:, 4], p_check
            )
            hit_any = jnp.any(hit_mask)
            hit_idx = jnp.argmax(hit_mask.astype(jnp.int32)).astype(jnp.int32)
            sun_dist = point_to_segment_distance(CENTER, CENTER, old_x, old_y, new_x, new_y)
            hit_sun = sun_dist < SUN_RADIUS
            in_bounds = (0.0 <= new_x) & (new_x <= BOARD_SIZE) & (0.0 <= new_y) & (new_y <= BOARD_SIZE)
            blocked = hit_sun | (~in_bounds)
            return hit_any, hit_idx, blocked

        hit_any_by_step, hit_idx_by_step, blocked_by_step = jax.vmap(step_hit)(steps)
        event_any = hit_any_by_step | blocked_by_step
        event_step = jnp.min(jnp.where(event_any, steps, V24_HORIZON + 1))
        planet_hit = hit_any_by_step & (steps == event_step)
        valid_hit = alive & jnp.any(planet_hit)
        idx_at = jnp.argmax(planet_hit.astype(jnp.int32)).astype(jnp.int32)
        target_idx = hit_idx_by_step[idx_at]
        return target_idx, event_step.astype(jnp.float32), valid_hit

    return jax.vmap(infer_one)(state.fleets)


def _incoming_to_targets_by_owner(state: JaxState):
    target_idx, hit_step, hit_valid = _fleet_first_contact_table(state)
    f_owner = state.fleets[:, 1].astype(jnp.int32)
    f_ships = state.fleets[:, 6]
    planet_ix = jnp.arange(MAX_PLANETS, dtype=jnp.int32)
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)

    def per_planet(pi):
        hits_pi = hit_valid & (target_idx == pi)
        return jax.vmap(lambda p: jnp.sum(jnp.where(hits_pi & (f_owner == p), f_ships, 0.0)))(players)

    by_owner = jax.vmap(per_planet)(planet_ix)
    soon_weight = jnp.clip(1.0 - (hit_step - 1.0) / jnp.maximum(V24_HORIZON, 1), 0.0, 1.0)

    def per_planet_pressure(pi):
        hits_pi = hit_valid & (target_idx == pi)
        return jax.vmap(lambda p: jnp.sum(jnp.where(hits_pi & (f_owner == p), f_ships * soon_weight, 0.0)))(players)

    pressure = jax.vmap(per_planet_pressure)(planet_ix)
    return by_owner, pressure, target_idx, hit_step, hit_valid


def _project_planet_owner_ships(state: JaxState, target_idx, player_id, extra_my_ships, eta):
    # Approximate V24 garrison projection using inferred fleet first-contact targets.
    planets = state.planets
    target = planets[target_idx]
    owner0 = target[1].astype(jnp.int32)
    ships0 = target[5]
    prod0 = target[6]
    turns = jnp.ceil(jnp.maximum(eta, 0.0))
    base_ships = ships0 + jnp.where(owner0 >= 0, prod0 * turns, 0.0)
    target_idx_by_fleet, hit_step, hit_valid = _fleet_first_contact_table(state)
    f_owner = state.fleets[:, 1].astype(jnp.int32)
    f_ships = state.fleets[:, 6]
    arrives = hit_valid & (target_idx_by_fleet == target_idx) & (hit_step <= turns + 1.0)
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)
    incoming = jax.vmap(lambda p: jnp.sum(jnp.where(arrives & (f_owner == p), f_ships, 0.0)))(players)
    incoming = incoming.at[player_id].add(extra_my_ships)
    garrison_by_owner = jnp.where(players == owner0, base_ships, 0.0) + incoming
    top_owner = jnp.argmax(garrison_by_owner).astype(jnp.int32)
    top_ships = garrison_by_owner[top_owner]
    second = jnp.max(jnp.where(players == top_owner, -1.0, garrison_by_owner))
    tie = jnp.sum((garrison_by_owner == top_ships) & (top_ships > 0.0)) > 1
    survivor = jnp.where(tie, 0.0, top_ships - jnp.maximum(second, 0.0))
    out_owner = jnp.where(survivor > 0.0, top_owner, owner0)
    out_ships = jnp.where(survivor > 0.0, survivor, base_ships)
    return out_owner, out_ships


def _resolve_garrison(owner, ships, incoming):
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)
    valid_owner = (owner >= 0) & (owner < MAX_PLAYERS)
    buckets = incoming + jnp.where(players == owner, ships, 0.0) * valid_owner.astype(jnp.float32)
    top_owner = jnp.argmax(buckets).astype(jnp.int32)
    top_ships = buckets[top_owner]
    second = jnp.max(jnp.where(players == top_owner, -1.0, buckets))
    tie = jnp.sum((buckets == top_ships) & (top_ships > 0.0)) > 1
    any_incoming = jnp.sum(incoming) > 0.0
    survivor = jnp.where(tie, 0.0, top_ships - jnp.maximum(second, 0.0))
    new_owner = jnp.where((survivor > 0.0) & (any_incoming | valid_owner), top_owner, owner)
    new_ships = jnp.where((survivor > 0.0) & (any_incoming | valid_owner), survivor, ships)
    return new_owner, new_ships


def _target_incoming_buckets(state: JaxState, target_idx):
    fleets = state.fleets
    f_owner = fleets[:, 1].astype(jnp.int32)
    f_ships = fleets[:, 6]
    target_idx_by_fleet, hit_step, hit_valid = _fleet_first_contact_table(state)
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)

    def bucket_for_step(k):
        at_k = hit_valid & (target_idx_by_fleet == target_idx) & (hit_step == k.astype(jnp.float32))
        return jax.vmap(lambda p: jnp.sum(jnp.where(at_k & (f_owner == p), f_ships, 0.0)))(players)

    return jax.vmap(bucket_for_step)(jnp.arange(1, V24_HORIZON + 1, dtype=jnp.int32))


def _target_flow_value(state: JaxState, target_idx, player_id, extra_ships, eta):
    planet = state.planets[target_idx]
    owner0 = planet[1].astype(jnp.int32)
    ships0 = planet[5]
    prod0 = planet[6]
    incoming = _target_incoming_buckets(state, target_idx)
    k_extra = jnp.clip(jnp.ceil(jnp.maximum(eta, 1.0)).astype(jnp.int32), 1, V24_HORIZON)
    incoming = incoming.at[k_extra - 1, player_id].add(extra_ships)

    def one_step(carry, k_incoming):
        owner, ships, value = carry
        pre_owner = owner
        ships = ships + jnp.where(owner >= 0, prod0, 0.0)
        owner, ships = _resolve_garrison(owner, ships, k_incoming)
        mine = owner == player_id
        enemy_owner = (owner >= 0) & (owner != player_id)
        was_mine = pre_owner == player_id
        prod_value = prod0 * mine.astype(jnp.float32) * 5.0
        enemy_denial = prod0 * enemy_owner.astype(jnp.float32) * -1.8
        ship_value = ships * mine.astype(jnp.float32) * 0.035
        flip_bonus = (mine & (~was_mine)).astype(jnp.float32) * (3.0 + prod0 * 1.5)
        hold_bonus = (mine & was_mine).astype(jnp.float32) * 0.25
        value = value + prod_value + enemy_denial + ship_value + flip_bonus + hold_bonus
        return (owner, ships, value), None

    (owner_f, ships_f, value), _ = lax.scan(one_step, (owner0, ships0, jnp.asarray(0.0, dtype=jnp.float32)), incoming)
    return value, owner_f, ships_f


def _planet_flow_value_with_initial_delta(state: JaxState, planet_idx, player_id, initial_delta):
    planet = state.planets[planet_idx]
    owner0 = planet[1].astype(jnp.int32)
    ships0 = jnp.maximum(0.0, planet[5] + initial_delta)
    prod0 = planet[6]
    incoming = _target_incoming_buckets(state, planet_idx)

    def one_step(carry, k_incoming):
        owner, ships, value = carry
        pre_owner = owner
        ships = ships + jnp.where(owner >= 0, prod0, 0.0)
        owner, ships = _resolve_garrison(owner, ships, k_incoming)
        mine = owner == player_id
        enemy_owner = (owner >= 0) & (owner != player_id)
        was_mine = pre_owner == player_id
        prod_value = prod0 * mine.astype(jnp.float32) * 5.0
        enemy_denial = prod0 * enemy_owner.astype(jnp.float32) * -1.8
        ship_value = ships * mine.astype(jnp.float32) * 0.035
        flip_bonus = (mine & (~was_mine)).astype(jnp.float32) * (3.0 + prod0 * 1.5)
        hold_bonus = (mine & was_mine).astype(jnp.float32) * 0.25
        value = value + prod_value + enemy_denial + ship_value + flip_bonus + hold_bonus
        return (owner, ships, value), None

    (owner_f, ships_f, value), _ = lax.scan(one_step, (owner0, ships0, jnp.asarray(0.0, dtype=jnp.float32)), incoming)
    return value, owner_f, ships_f


def _source_debit_delta_score(state: JaxState, source_idx, player_id, send):
    base_value, base_owner, base_ships = _planet_flow_value_with_initial_delta(state, source_idx, player_id, 0.0)
    hyp_value, hyp_owner, hyp_ships = _planet_flow_value_with_initial_delta(state, source_idx, player_id, -send)
    lose_owner_penalty = ((base_owner == player_id) & (hyp_owner != player_id)).astype(jnp.float32) * -6.0
    ship_delta = (hyp_ships - base_ships) * (hyp_owner == player_id).astype(jnp.float32) * 0.025
    return (hyp_value - base_value) + lose_owner_penalty + ship_delta


def _target_flow_delta_score(state: JaxState, target_idx, player_id, send_total, eta, is_defense):
    base_value, base_owner, base_ships = _target_flow_value(state, target_idx, player_id, 0.0, eta)
    hyp_value, hyp_owner, hyp_ships = _target_flow_value(state, target_idx, player_id, send_total, eta)
    delta = hyp_value - base_value
    spend_penalty = send_total * jnp.where(is_defense, 0.010, 0.030)
    final_bonus = (hyp_owner == player_id).astype(jnp.float32) * 1.5 - ((base_owner == player_id) & (hyp_owner != player_id)).astype(jnp.float32) * 3.0
    ship_delta = (hyp_ships - base_ships) * (hyp_owner == player_id).astype(jnp.float32) * 0.025
    return delta + final_bonus + ship_delta - spend_penalty


def _flow_score_launch(state: JaxState, source_idx, target_idx, player_id, send, eta, is_defense):
    target_delta = _target_flow_delta_score(state, target_idx, player_id, send, eta, is_defense)
    source_delta = _source_debit_delta_score(state, source_idx, player_id, send)
    return target_delta + source_delta


def _flow_score_multi_launch(state: JaxState, source_idx, target_idx, player_id, sends, active, eta, is_defense):
    total_send = jnp.sum(jnp.where(active, sends, 0.0))
    target_delta = _target_flow_delta_score(state, target_idx, player_id, total_send, eta, is_defense)
    source_delta = jnp.sum(jax.vmap(lambda sidx, snd, ok: jnp.where(ok, _source_debit_delta_score(state, sidx, player_id, snd), 0.0))(source_idx, sends, active))
    return target_delta + source_delta


def _flow_score_target(state: JaxState, target_idx, player_id, send_total, eta, is_defense):
    return _target_flow_delta_score(state, target_idx, player_id, send_total, eta, is_defense)


def _first_contact_is_target(state: JaxState, src_idx, tgt_idx, angle, send, eta):
    # Approximate V24 body screen: simulate swept contact for this straight shot.
    planets = state.planets
    alive = planets[:, 0] >= 0
    speed = fleet_speed(jnp.maximum(send, 1.0), 6.0)
    start_x = planets[src_idx, 2] + jnp.cos(angle) * (planets[src_idx, 4] + 0.1)
    start_y = planets[src_idx, 3] + jnp.sin(angle) * (planets[src_idx, 4] + 0.1)
    steps = jnp.arange(1, V24_HORIZON + 1, dtype=jnp.float32)
    old_t = steps - 1.0
    new_t = steps
    fx0 = start_x + jnp.cos(angle) * speed * old_t
    fy0 = start_y + jnp.sin(angle) * speed * old_t
    fx1 = start_x + jnp.cos(angle) * speed * new_t
    fy1 = start_y + jnp.sin(angle) * speed * new_t

    def hit_for_step(k_idx):
        k0 = k_idx.astype(jnp.float32)
        px0, py0 = _planet_xy_at_step(state, k0)
        px1, py1 = _planet_xy_at_step(state, k0 + 1.0)
        f0x = fx0[k_idx]
        f0y = fy0[k_idx]
        f1x = fx1[k_idx]
        f1y = fy1[k_idx]
        hits = jax.vmap(lambda ax, ay, bx, by, r, ok: ok & swept_pair_hit(f0x, f0y, f1x, f1y, ax, ay, bx, by, r))(
            px0, py0, px1, py1, planets[:, 4], alive
        )
        hit_any = jnp.any(hits)
        first_idx = jnp.argmax(hits.astype(jnp.int32)).astype(jnp.int32)
        return hit_any, first_idx

    hit_any_by_step, first_idx_by_step = jax.vmap(hit_for_step)(jnp.arange(V24_HORIZON, dtype=jnp.int32))
    step_numbers = jnp.arange(1, V24_HORIZON + 1, dtype=jnp.float32)
    first_hit_step = jnp.min(jnp.where(hit_any_by_step, step_numbers, 999.0))
    first_hit_pos = jnp.argmax(hit_any_by_step.astype(jnp.int32)).astype(jnp.int32)
    first_planet = first_idx_by_step[first_hit_pos]

    in_bounds = (fx1 >= 0.0) & (fx1 <= BOARD_SIZE) & (fy1 >= 0.0) & (fy1 <= BOARD_SIZE)
    sun_dist = jax.vmap(lambda ax, ay, bx, by: point_to_segment_distance(CENTER, CENTER, ax, ay, bx, by))(fx0, fy0, fx1, fy1)
    killed = (~in_bounds) | (sun_dist < SUN_RADIUS)
    first_kill_step = jnp.min(jnp.where(killed, step_numbers, 999.0))
    eta_step = jnp.ceil(jnp.maximum(eta, 1.0))
    target_hit = first_hit_step <= jnp.minimum(first_kill_step, eta_step + 1.0)
    return target_hit & (first_planet == tgt_idx)


def _candidate_feature_vector(base_score, target_prod, target_ships, source_ships, total_send, min_eta, max_eta, active_count, neutral, enemy, defense, step):
    return jnp.stack([
        base_score / 50.0,
        target_prod / 8.0,
        target_ships / 200.0,
        source_ships / 200.0,
        total_send / 200.0,
        min_eta / 20.0,
        max_eta / 20.0,
        active_count / 4.0,
        neutral,
        enemy,
        defense,
        jnp.ones_like(base_score) * step.astype(jnp.float32) / 500.0,
    ], axis=-1)


def v24_like_candidates_for_player(state: JaxState, player_id):
    planets = state.planets
    ids = planets[:, 0].astype(jnp.int32)
    owner = planets[:, 1].astype(jnp.int32)
    x = planets[:, 2]
    y = planets[:, 3]
    radius = planets[:, 4]
    ships = planets[:, 5]
    prod = planets[:, 6]
    alive = ids >= 0
    valid_player = player_id < state.player_count
    is_comet = alive & (ids >= state.comet_base_id) & (ids < state.comet_base_id + 4)
    owned = alive & (owner == player_id) & (~is_comet)
    enemy = alive & (owner >= 0) & (owner != player_id) & (~is_comet)
    neutral = alive & (owner < 0) & (~is_comet)
    attack_mask = enemy | neutral

    # V24-style source shortlist: strongest owned planets that can launch.
    source_mask = owned & (ships >= V24_MIN_SHIPS_TO_LAUNCH)
    src_idx, src_exists, _ = _topk_masked(ships, source_mask, V24_MAX_SOURCES)
    src_x = x[src_idx]
    src_y = y[src_idx]
    src_ships = ships[src_idx]
    src_prod = prod[src_idx]
    f_alive = state.fleets[:, 0] >= 0
    f_owner = state.fleets[:, 1].astype(jnp.int32)
    f_enemy = f_alive & (f_owner >= 0) & (f_owner != player_id)
    fdx = state.fleets[:, 2:3] - src_x[None, :]
    fdy = state.fleets[:, 3:4] - src_y[None, :]
    fd = jnp.sqrt(fdx * fdx + fdy * fdy) + 1e-6
    near_pressure = jnp.sum(jnp.where(f_enemy[:, None], state.fleets[:, 6:7] * jnp.clip(1.0 - fd / 32.0, 0.0, 1.0), 0.0), axis=0)
    _, target_pressure, _, _, _ = _incoming_to_targets_by_owner(state)
    source_target_pressure = jax.vmap(lambda s: jnp.sum(jnp.where(jnp.arange(MAX_PLAYERS, dtype=jnp.int32) != player_id, target_pressure[s], 0.0)))(src_idx)
    incoming_pressure = near_pressure * 0.45 + source_target_pressure
    reserve = V24_SAFE_RESERVE_BASE + src_prod + incoming_pressure * 0.45
    safe_drain_raw = jnp.maximum(0.0, jnp.floor(src_ships * V24_SAFE_DRAIN_FRAC - reserve))

    def flow_guarded_drain(sidx, raw, reserve_src, exists):
        base_value, base_owner, base_ships = _planet_flow_value_with_initial_delta(state, sidx, player_id, 0.0)
        hyp_value, hyp_owner, hyp_ships = _planet_flow_value_with_initial_delta(state, sidx, player_id, -raw)
        still_mine = (base_owner == player_id) & (hyp_owner == player_id)
        weak_after = hyp_ships < jnp.maximum(2.0, state.planets[sidx, 6] + 1.0)
        value_loss = (base_value - hyp_value) > 3.5
        capped = jnp.maximum(0.0, jnp.minimum(raw * 0.55, state.planets[sidx, 5] - reserve_src - 2.0))
        guarded = jnp.where(still_mine & (~weak_after) & (~value_loss), raw, capped)
        return jnp.where(exists, jnp.floor(jnp.maximum(0.0, guarded)), 0.0)

    safe_drain = jax.vmap(flow_guarded_drain)(src_idx, safe_drain_raw, reserve, src_exists)

    # Attack shortlist: target selected by closest source plus production/value.
    dx_st = x[None, :] - src_x[:, None]
    dy_st = y[None, :] - src_y[:, None]
    dist_st = jnp.sqrt(dx_st * dx_st + dy_st * dy_st) + 1e-6
    min_src_dist = jnp.min(jnp.where(src_exists[:, None], dist_st, 1.0e6), axis=0)
    target_pref = -min_src_dist + prod * 3.0 - ships * 0.05 + neutral.astype(jnp.float32) * 5.0 + enemy.astype(jnp.float32) * 3.0
    atk_idx, atk_exists, _ = _topk_masked(target_pref, attack_mask, V24_MAX_ATTACK_TARGETS)

    # Defense/regroup shortlist: own planets with high value and low garrison.
    # This approximates V24 friendly_flip_targets + cheap_enemy_pressure without full garrison cache.
    enemy_dx = x[:, None] - x[None, :]
    enemy_dy = y[:, None] - y[None, :]
    all_dist = jnp.sqrt(enemy_dx * enemy_dx + enemy_dy * enemy_dy) + 1e-6
    enemy_pressure = jnp.sum(jnp.where(enemy[:, None], ships[:, None] * jnp.clip(1.0 - all_dist / 55.0, 0.0, 1.0), 0.0), axis=0)
    def_pref = prod * 3.0 + enemy_pressure * 0.15 - ships * 0.10
    def_idx, def_exists, _ = _topk_masked(def_pref, owned, V24_MAX_DEF_TARGETS)

    # Fixed candidate arrays.
    C_SINGLE = V24_MAX_SOURCES * V24_MAX_ATTACK_TARGETS
    C_FOCUS = V24_MAX_ATTACK_TARGETS
    C_REGROUP = V24_MAX_SOURCES * V24_MAX_DEF_TARGETS
    C_GREEDY = 1
    C_TOTAL = C_SINGLE + C_FOCUS + C_REGROUP + C_GREEDY
    moves = jnp.zeros((NUM_CANDIDATES, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32)
    features = jnp.zeros((NUM_CANDIDATES, FEATURE_DIM), dtype=jnp.float32)
    valid = jnp.zeros((NUM_CANDIDATES,), dtype=jnp.bool_)
    base_score = jnp.full((NUM_CANDIDATES,), -1.0e9, dtype=jnp.float32)

    # Single-source attack/reinforcement wave candidates.
    s_grid = jnp.repeat(jnp.arange(V24_MAX_SOURCES, dtype=jnp.int32), V24_MAX_ATTACK_TARGETS)
    t_grid = jnp.tile(jnp.arange(V24_MAX_ATTACK_TARGETS, dtype=jnp.int32), V24_MAX_SOURCES)
    si = src_idx[s_grid]
    ti = atk_idx[t_grid]
    tx0 = x[ti]; ty0 = y[ti]
    send = safe_drain[s_grid]
    spd = jnp.maximum(fleet_speed(jnp.maximum(send, 1.0), 6.0), 1.0)
    dist0 = jnp.sqrt((tx0 - x[si]) ** 2 + (ty0 - y[si]) ** 2) + 1e-6
    eta0 = dist0 / spd
    tx, ty = _future_planet_xy(state, ti, eta0)
    dist = jnp.sqrt((tx - x[si]) ** 2 + (ty - y[si]) ** 2) + 1e-6
    eta = dist / spd
    need = _projected_capture_need(owner[ti], ships[ti], prod[ti], eta)
    angle = _angle_between(x[si], y[si], tx, ty)
    clear_sun = _sun_line_clear(x[si], y[si], tx, ty)
    body_ok = jax.vmap(lambda a, b, c, d, e: _first_contact_is_target(state, a, b, c, d, e))(si, ti, angle, send, eta)
    single_valid = valid_player & src_exists[s_grid] & atk_exists[t_grid] & (send >= V24_MIN_SHIPS_TO_LAUNCH) & (send >= need) & (eta <= V24_HORIZON) & clear_sun & body_ok
    flow_single = jax.vmap(lambda src, t, snt, e: _flow_score_launch(state, src, t, player_id, snt, e, False))(si, ti, send, eta)
    single_score = flow_single - eta * 0.22 + prod[ti] * 0.45
    single_moves = jnp.zeros((C_SINGLE, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32).at[:, 0, :].set(jnp.stack([ids[si].astype(jnp.float32), angle, send], axis=-1))
    single_feat = _candidate_feature_vector(single_score, prod[ti], ships[ti], ships[si], send, eta, eta, jnp.ones_like(send), neutral[ti].astype(jnp.float32), enemy[ti].astype(jnp.float32), jnp.zeros_like(send), state.step)
    moves = moves.at[:C_SINGLE].set(single_moves)
    features = features.at[:C_SINGLE].set(single_feat)
    valid = valid.at[:C_SINGLE].set(single_valid)
    base_score = base_score.at[:C_SINGLE].set(jnp.where(single_valid, single_score, -1.0e9))

    # Greedy multi-wave candidate: closer V24 _greedy_select over single-source candidates.
    # Includes source budget, one wave per target, target-as-source mutex, and lookahead bonus.
    def greedy_scan_step(carry, widx):
        budget, taken, used_src, gmoves, gscore, gsend, gmin_eta, gmax_eta, gn = carry
        can_fund = budget[si] >= send
        not_taken = ~taken[t_grid]
        target_used_as_src = used_src[ti]
        mask = single_valid & can_fund & not_taken & (~target_used_as_src) & (single_score > V24_ROI_THRESHOLD)
        current = jnp.where(mask, single_score, -1.0e9)

        # V24-like one-step lookahead: value of best next candidate after selecting i.
        budget_after_i = jnp.maximum(0.0, budget[None, :] - jax.nn.one_hot(si, MAX_PLANETS) * send[:, None])
        j_budget = budget_after_i[:, si]
        j_can_fund = send[None, :] <= j_budget
        i_takes = t_grid[:, None] == t_grid[None, :]
        i_used_src = jax.nn.one_hot(si, MAX_PLANETS).astype(jnp.bool_)
        j_target_used_by_i = i_used_src[:, ti]
        j_not_taken = ~taken[t_grid][None, :]
        j_mask = mask[None, :] & j_can_fund & (~i_takes) & (~j_target_used_by_i) & j_not_taken
        next_best = jnp.max(jnp.where(j_mask, single_score[None, :], -1.0e9), axis=1)
        next_bonus = jnp.maximum(0.0, next_best - V24_ROI_THRESHOLD)
        ranked = jnp.where(mask, current + 0.25 * next_bonus, -1.0e9)

        best = jnp.argmax(ranked).astype(jnp.int32)
        best_score = current[best]
        fired = best_score > -1.0e8
        bsrc = si[best]
        btpos = t_grid[best]
        btgt = ti[best]
        bsend = send[best]
        brow = jnp.stack([ids[bsrc].astype(jnp.float32), angle[best], bsend], axis=-1)
        gmoves = gmoves.at[widx].set(jnp.where(fired, brow, jnp.zeros((3,), dtype=jnp.float32)))
        budget = budget.at[bsrc].add(jnp.where(fired, -bsend, 0.0))
        taken = taken.at[btpos].set(taken[btpos] | fired)
        used_src = used_src.at[bsrc].set(used_src[bsrc] | fired)
        # Target cannot later act as a source in this same greedy turn.
        used_src = used_src.at[btgt].set(used_src[btgt] | fired)
        gscore = gscore + jnp.where(fired, best_score, 0.0)
        gsend = gsend + jnp.where(fired, bsend, 0.0)
        geta = eta[best]
        gmin_eta = jnp.minimum(gmin_eta, jnp.where(fired, geta, 99.0))
        gmax_eta = jnp.maximum(gmax_eta, jnp.where(fired, geta, 0.0))
        gn = gn + fired.astype(jnp.float32)
        return (budget, taken, used_src, gmoves, gscore, gsend, gmin_eta, gmax_eta, gn), None

    init_gmoves = jnp.zeros((MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32)
    init_taken = jnp.zeros((V24_MAX_ATTACK_TARGETS,), dtype=jnp.bool_)
    init_used_src = jnp.zeros((MAX_PLANETS,), dtype=jnp.bool_)
    (gbudget, gtaken, gused_src, gmoves, gscore, gsend, gmin_eta, gmax_eta, gn), _ = lax.scan(
        greedy_scan_step,
        (ships, init_taken, init_used_src, init_gmoves, jnp.asarray(0.0, dtype=jnp.float32), jnp.asarray(0.0, dtype=jnp.float32), jnp.asarray(99.0, dtype=jnp.float32), jnp.asarray(0.0, dtype=jnp.float32), jnp.asarray(0.0, dtype=jnp.float32)),
        jnp.arange(V24_MAX_WAVES, dtype=jnp.int32),
    )
    greedy_valid = valid_player & (gn > 0.0)
    greedy_score = gscore + gn * 0.35
    best_single = jnp.argmax(jnp.where(single_valid, single_score, -1.0e9)).astype(jnp.int32)
    greedy_feat = _candidate_feature_vector(
        greedy_score,
        prod[ti[best_single]],
        ships[ti[best_single]],
        ships[si[best_single]],
        gsend,
        gmin_eta,
        gmax_eta,
        gn,
        neutral[ti[best_single]].astype(jnp.float32),
        enemy[ti[best_single]].astype(jnp.float32),
        jnp.zeros((), dtype=jnp.float32),
        state.step,
    )

    # Focus-fire candidates: top strike sources combine into same target.
    def focus_for_target(tpos):
        ti = atk_idx[tpos]
        tx0 = x[ti]; ty0 = y[ti]
        spd_s = jnp.maximum(fleet_speed(jnp.maximum(safe_drain, 1.0), 6.0), 1.0)
        d0 = jnp.sqrt((tx0 - src_x) ** 2 + (ty0 - src_y) ** 2) + 1e-6
        eta0_s = d0 / spd_s
        txv, tyv = _future_planet_xy(state, ti, eta0_s)
        d = jnp.sqrt((txv - src_x) ** 2 + (tyv - src_y) ** 2) + 1e-6
        eta_s = d / spd_s
        clear_sun_s = _sun_line_clear(src_x, src_y, txv, tyv)
        body_ok_s = jax.vmap(lambda a, c, d, e: _first_contact_is_target(state, a, ti, c, d, e))(src_idx, _angle_between(src_x, src_y, txv, tyv), safe_drain, eta_s)
        contributor_score = safe_drain - eta_s * 3.0
        order_vals, order = lax.top_k(jnp.where(src_exists & (safe_drain >= V24_MIN_SHIPS_TO_LAUNCH) & (eta_s <= V24_HORIZON) & clear_sun_s & body_ok_s, contributor_score, -1.0e9), V24_MAX_STRIKE_SOURCES)
        chosen_src = src_idx[order]
        chosen_send = safe_drain[order]
        chosen_eta = eta_s[order]
        chosen_tx = txv[order]
        chosen_ty = tyv[order]
        chosen_angle = _angle_between(x[chosen_src], y[chosen_src], chosen_tx, chosen_ty)
        active = order_vals > -1.0e8
        total = jnp.sum(jnp.where(active, chosen_send, 0.0))
        min_e = jnp.min(jnp.where(active, chosen_eta, 99.0))
        max_e = jnp.max(jnp.where(active, chosen_eta, 0.0))
        active_n = jnp.sum(active.astype(jnp.float32))
        flow_f = _flow_score_multi_launch(state, chosen_src, ti, player_id, chosen_send, active, max_e, False)
        score_f = flow_f - max_e * 0.22 + prod[ti] * 0.55
        mv = jnp.zeros((MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32)
        rows = jnp.stack([ids[chosen_src].astype(jnp.float32), chosen_angle, chosen_send], axis=-1)
        mv = mv.at[:V24_MAX_STRIKE_SOURCES].set(jnp.where(active[:, None], rows, 0.0))
        need_f = _projected_capture_need(owner[ti], ships[ti], prod[ti], max_e)
        ok = valid_player & atk_exists[tpos] & (active_n >= 2.0) & (total >= need_f) & (max_e <= V24_HORIZON)
        feat = _candidate_feature_vector(score_f, prod[ti], ships[ti], jnp.max(jnp.where(active, ships[chosen_src], 0.0)), total, min_e, max_e, active_n, neutral[ti].astype(jnp.float32), enemy[ti].astype(jnp.float32), jnp.zeros((), dtype=jnp.float32), state.step)
        return mv, feat, ok, jnp.where(ok, score_f, -1.0e9)
    f_moves, f_feat, f_valid, f_score = jax.vmap(focus_for_target)(jnp.arange(V24_MAX_ATTACK_TARGETS, dtype=jnp.int32))
    fs = C_SINGLE
    moves = moves.at[fs:fs+C_FOCUS].set(f_moves)
    features = features.at[fs:fs+C_FOCUS].set(f_feat)
    valid = valid.at[fs:fs+C_FOCUS].set(f_valid)
    base_score = base_score.at[fs:fs+C_FOCUS].set(f_score)

    # Regroup/defense candidates.
    rs_grid = jnp.repeat(jnp.arange(V24_MAX_SOURCES, dtype=jnp.int32), V24_MAX_DEF_TARGETS)
    rt_grid = jnp.tile(jnp.arange(V24_MAX_DEF_TARGETS, dtype=jnp.int32), V24_MAX_SOURCES)
    rsi = src_idx[rs_grid]
    rti = def_idx[rt_grid]
    rsend = jnp.maximum(V24_MIN_SHIPS_TO_LAUNCH, jnp.floor(safe_drain[rs_grid] * V24_REGROUP_SEND_FRAC))
    rspd = jnp.maximum(fleet_speed(jnp.maximum(rsend, 1.0), 6.0), 1.0)
    rdist0 = jnp.sqrt((x[rti] - x[rsi]) ** 2 + (y[rti] - y[rsi]) ** 2) + 1e-6
    reta0 = rdist0 / rspd
    rtx, rty = _future_planet_xy(state, rti, reta0)
    rdist = jnp.sqrt((rtx - x[rsi]) ** 2 + (rty - y[rsi]) ** 2) + 1e-6
    reta = rdist / rspd
    rangle = _angle_between(x[rsi], y[rsi], rtx, rty)
    rclear_sun = _sun_line_clear(x[rsi], y[rsi], rtx, rty)
    rbody_ok = jax.vmap(lambda a, b, c, d, e: _first_contact_is_target(state, a, b, c, d, e))(rsi, rti, rangle, rsend, reta)
    rflow = jax.vmap(lambda src, t, snt, e: _flow_score_launch(state, src, t, player_id, snt, e, True))(rsi, rti, rsend, reta)
    rscore = rflow + enemy_pressure[rti] * 0.10 - reta * 0.20
    rvalid = valid_player & src_exists[rs_grid] & def_exists[rt_grid] & (rsi != rti) & (safe_drain[rs_grid] >= V24_MIN_SHIPS_TO_LAUNCH) & (reta <= V24_HORIZON) & rclear_sun & rbody_ok
    rmoves = jnp.zeros((C_REGROUP, MAX_MOVES_PER_PLAYER, 3), dtype=jnp.float32).at[:, 0, :].set(jnp.stack([ids[rsi].astype(jnp.float32), rangle, rsend], axis=-1))
    rfeat = _candidate_feature_vector(rscore, prod[rti], ships[rti], ships[rsi], rsend, reta, reta, jnp.ones_like(rsend), jnp.zeros_like(rsend), jnp.zeros_like(rsend), jnp.ones_like(rsend), state.step)
    rs0 = C_SINGLE + C_FOCUS
    moves = moves.at[rs0:rs0+C_REGROUP].set(rmoves)
    features = features.at[rs0:rs0+C_REGROUP].set(rfeat)
    valid = valid.at[rs0:rs0+C_REGROUP].set(rvalid)
    base_score = base_score.at[rs0:rs0+C_REGROUP].set(jnp.where(rvalid, rscore, -1.0e9))

    gs0 = C_SINGLE + C_FOCUS + C_REGROUP
    moves = moves.at[gs0].set(gmoves)
    features = features.at[gs0].set(greedy_feat)
    valid = valid.at[gs0].set(greedy_valid)
    base_score = base_score.at[gs0].set(jnp.where(greedy_valid, greedy_score, -1.0e9))

    return moves, features, valid, base_score


def build_candidates_single(state: JaxState):
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)
    moves, feat, valid, base_score = jax.vmap(lambda p: v24_like_candidates_for_player(state, p))(players)
    return moves, feat, valid, base_score


def policy_logits_values(params, features, valid, base_score=None):
    h = jnp.tanh(features @ params.w1 + params.b1)
    learned = (h @ params.w2 + params.b2).squeeze(-1) * BIAS_SCALE
    prior = 0.0 if base_score is None else base_score / 10.0
    logits = learned + prior
    logits = jnp.where(valid, logits, -1.0e9)
    any_valid = jnp.any(valid, axis=-1, keepdims=True)
    fallback = jnp.concatenate([jnp.zeros_like(logits[..., :1]), jnp.full_like(logits[..., 1:], -1.0e9)], axis=-1)
    logits = jnp.where(any_valid, logits, fallback)
    pooled = jnp.mean(features, axis=-2)
    vh = jnp.tanh(pooled @ params.value_w1 + params.value_b1)
    values = (vh @ params.value_w2 + params.value_b2).squeeze(-1)
    return logits, values


def sample_actions(params, state, key, temperature=TRAIN_TEMPERATURE):
    cand_moves, features, valid, base_score = build_candidates_single(state)
    logits, values = policy_logits_values(params, features, valid, base_score)
    keys = jrandom.split(key, MAX_PLAYERS)
    chosen = jax.vmap(lambda k, lg: jrandom.categorical(k, lg / temperature))(keys, logits)
    chosen_moves = jnp.take_along_axis(cand_moves, chosen[:, None, None, None], axis=1).squeeze(1)
    has_valid = jnp.any(valid, axis=-1)
    chosen_moves = jnp.where(has_valid[:, None, None], chosen_moves, jnp.zeros_like(chosen_moves))
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    chosen_logp = jnp.take_along_axis(log_probs, chosen[:, None], axis=-1).squeeze(-1)
    probs = jax.nn.softmax(logits, axis=-1)
    entropy = -jnp.sum(probs * log_probs, axis=-1)
    return JaxAction(moves=chosen_moves), features, valid, base_score, chosen, chosen_logp, values, entropy

batched_sample_actions = jax.jit(jax.vmap(sample_actions, in_axes=(None, 0, 0, None)))


def strategic_score_single(state: JaxState):
    planets = state.planets
    fleets = state.fleets
    p_alive = planets[:, 0] >= 0
    f_alive = fleets[:, 0] >= 0
    p_owner = planets[:, 1].astype(jnp.int32)
    f_owner = fleets[:, 1].astype(jnp.int32)
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)
    valid_player = players < state.player_count

    prod_score = jax.vmap(lambda pl: jnp.sum(jnp.where(p_alive & (p_owner == pl), planets[:, 6], 0.0)))(players)
    planet_count = jax.vmap(lambda pl: jnp.sum((p_alive & (p_owner == pl)).astype(jnp.float32)))(players)
    planet_ships = jax.vmap(lambda pl: jnp.sum(jnp.where(p_alive & (p_owner == pl), planets[:, 5], 0.0)))(players)
    fleet_ships = jax.vmap(lambda pl: jnp.sum(jnp.where(f_alive & (f_owner == pl), fleets[:, 6], 0.0)))(players)

    raw = (
        SHAPE_PROD_WEIGHT * prod_score
        + SHAPE_PLANET_WEIGHT * planet_count
        + SHAPE_PLANET_SHIP_WEIGHT * planet_ships
        + SHAPE_FLEET_SHIP_WEIGHT * fleet_ships
    )
    valid_count = jnp.maximum(jnp.sum(valid_player.astype(jnp.float32)), 2.0)
    opp_mean = (jnp.sum(jnp.where(valid_player, raw, 0.0)) - raw) / (valid_count - 1.0)
    rel = raw - opp_mean
    lead_term = SHAPE_LEAD_WEIGHT * jnp.tanh(rel / 25.0)
    score = raw + lead_term
    return jnp.where(valid_player, score, 0.0)


def shaped_reward_single(prev_state: JaxState, next_state: JaxState, env_reward):
    prev_score = strategic_score_single(prev_state)
    next_score = strategic_score_single(next_state)
    dense = (next_score - prev_score) * SHAPING_REWARD_SCALE
    dense = jnp.clip(dense, -SHAPING_REWARD_CLIP, SHAPING_REWARD_CLIP)
    total = TERMINAL_REWARD_WEIGHT * env_reward + dense
    return jnp.clip(total, -TOTAL_REWARD_CLIP, TOTAL_REWARD_CLIP)


batched_shaped_reward = jax.jit(jax.vmap(shaped_reward_single, in_axes=(0, 0, 0)))


def _where_batch(mask, yes, no):
    mask = mask.astype(jnp.bool_)
    while mask.ndim < yes.ndim:
        mask = mask[..., None]
    return jnp.where(mask, yes, no)


def reset_done_state(prev_state: JaxState, reset_state: JaxState) -> JaxState:
    # Neu env da ket thuc tu buoc truoc, buoc tiep theo bat dau mot tran moi.
    # Terminal reward da duoc ghi o transition truoc do, nen khong bi lap lai.
    use_reset = prev_state.done
    return JaxState(
        planets=_where_batch(use_reset, reset_state.planets, prev_state.planets),
        fleets=_where_batch(use_reset, reset_state.fleets, prev_state.fleets),
        initial_planets=_where_batch(use_reset, reset_state.initial_planets, prev_state.initial_planets),
        comet_paths=_where_batch(use_reset, reset_state.comet_paths, prev_state.comet_paths),
        comet_lengths=_where_batch(use_reset, reset_state.comet_lengths, prev_state.comet_lengths),
        comet_ships=_where_batch(use_reset, reset_state.comet_ships, prev_state.comet_ships),
        comet_base_id=_where_batch(use_reset, reset_state.comet_base_id, prev_state.comet_base_id),
        step=_where_batch(use_reset, reset_state.step, prev_state.step),
        next_fleet_id=_where_batch(use_reset, reset_state.next_fleet_id, prev_state.next_fleet_id),
        angular_velocity=_where_batch(use_reset, reset_state.angular_velocity, prev_state.angular_velocity),
        player_count=_where_batch(use_reset, reset_state.player_count, prev_state.player_count),
        done=jnp.where(use_reset, reset_state.done, prev_state.done),
        rewards=_where_batch(use_reset, reset_state.rewards, prev_state.rewards),
    )


def rollout_once(params, state, key, reset_state):
    players = jnp.arange(MAX_PLAYERS, dtype=jnp.int32)

    def one_step(carry, _):
        st, k = carry
        st = reset_done_state(st, reset_state)
        k, ak = jrandom.split(k)
        action, features, valid, base_score, chosen, logp, value, entropy = batched_sample_actions(params, st, jrandom.split(ak, st.planets.shape[0]), TRAIN_TEMPERATURE)
        next_st = jax_step(st, action, 6.0, 500)
        reward = batched_shaped_reward(st, next_st, next_st.rewards)
        valid_player = players[None, :] < st.player_count[:, None]
        active_sample = ((~st.done)[:, None]) & valid_player
        reward = jnp.where(active_sample, reward, jnp.zeros_like(reward))
        done = next_st.done
        data = (features, valid, base_score, chosen, logp, value, reward, done, entropy, active_sample)
        return (next_st, k), data
    (next_state, key), data = lax.scan(one_step, (state, key), None, length=ROLLOUT_STEPS)
    return next_state, key, data


def compute_gae(rewards, values, dones, last_values, active_mask):
    active_f = active_mask.astype(jnp.float32)
    values = jnp.where(active_mask, values, jnp.zeros_like(values))
    rewards = jnp.where(active_mask, rewards, jnp.zeros_like(rewards))
    def scan_fn(carry, x):
        gae, next_value = carry
        reward, value, done, active = x
        mask = 1.0 - done.astype(jnp.float32)[:, None]
        delta = reward + GAMMA * next_value * mask - value
        gae = delta + GAMMA * GAE_LAMBDA * mask * gae
        gae = jnp.where(active, gae, jnp.zeros_like(gae))
        return (gae, value), gae
    (_, _), adv_rev = lax.scan(scan_fn, (jnp.zeros_like(last_values), last_values), (rewards[::-1], values[::-1], dones[::-1], active_mask[::-1]))
    adv = adv_rev[::-1]
    returns = jnp.where(active_mask, adv + values, jnp.zeros_like(values))
    denom = active_f.sum().clip(min=1.0)
    mean = (adv * active_f).sum() / denom
    var = (((adv - mean) ** 2) * active_f).sum() / denom
    adv = jnp.where(active_mask, (adv - mean) / jnp.sqrt(var + 1e-6), jnp.zeros_like(adv))
    return adv, returns


def _masked_mean(x, mask):
    w = mask.astype(x.dtype)
    return (x * w).sum() / w.sum().clip(min=1.0)


def ppo_loss(params, batch):
    features, valid, base_score, chosen, old_logp, old_value, returns, advantages, active_mask = batch
    logits, values = policy_logits_values(params, features, valid, base_score)
    log_probs = jax.nn.log_softmax(logits, axis=-1)
    new_logp = jnp.take_along_axis(log_probs, chosen[..., None], axis=-1).squeeze(-1)
    ratio = jnp.exp(new_logp - old_logp)
    unclipped = ratio * advantages
    clipped = jnp.clip(ratio, 1.0 - PPO_CLIP_EPS, 1.0 + PPO_CLIP_EPS) * advantages
    policy_loss = -_masked_mean(jnp.minimum(unclipped, clipped), active_mask)
    value_loss = _masked_mean((returns - values) ** 2, active_mask)
    probs = jax.nn.softmax(logits, axis=-1)
    entropy_per = -jnp.sum(probs * log_probs, axis=-1)
    entropy = _masked_mean(entropy_per, active_mask)
    loss = policy_loss + VALUE_COEF * value_loss - ENTROPY_COEF * entropy
    return loss, (policy_loss, value_loss, entropy)

@jax.jit
def ppo_update(params, opt_state, batch):
    (loss, aux), grads = jax.value_and_grad(ppo_loss, has_aux=True)(params, batch)
    params, opt_state = adam_update(params, grads, opt_state)
    return params, opt_state, loss, aux


def flatten_ppo_batch(batch):
    # Flatten rollout dims [T, B, player] -> [samples]. Candidate dims stay intact.
    features, valid, base_score, chosen, old_logp, old_value, returns, advantages, active_mask = batch
    sample_n = features.shape[0] * features.shape[1] * features.shape[2]
    return (
        features.reshape((sample_n,) + features.shape[3:]),
        valid.reshape((sample_n,) + valid.shape[3:]),
        base_score.reshape((sample_n,) + base_score.shape[3:]),
        chosen.reshape((sample_n,)),
        old_logp.reshape((sample_n,)),
        old_value.reshape((sample_n,)),
        returns.reshape((sample_n,)),
        advantages.reshape((sample_n,)),
        active_mask.reshape((sample_n,)),
    )


def minibatch_slice(batch, start, size):
    return tuple(x[start:start + size] for x in batch)


def train_ppo(seed=0, updates=TOTAL_PPO_UPDATES, num_envs=NUM_ENVS, initial_params=None, start_update=0):
    key = jrandom.PRNGKey(seed + int(start_update) * 9973)
    params = init_policy_params(key) if initial_params is None else initial_params
    opt_state = init_adam(params)
    state = make_initial_jax_batch(seed_start=seed + 100 + int(start_update) * 17, batch_size=num_envs)
    # Prebuild a reset batch. Creating official maps is CPU/Python-heavy, so do not
    # regenerate it every PPO update; refresh it only when the full batch is reset.
    reset_state = make_initial_jax_batch(seed_start=seed + 200000 + int(start_update) * 31, batch_size=num_envs)
    history = []
    pbar = tqdm(range(int(updates)), total=int(updates), desc='PPO train')
    for update in pbar:
        key, rk = jrandom.split(key)
        state, key, data = rollout_once(params, state, rk, reset_state)
        features, valid, base_score, chosen, logp, values, rewards, dones, entropy, active_mask = data
        _, _, _, _, _, _, last_values, _ = batched_sample_actions(params, state, jrandom.split(key, num_envs), TRAIN_TEMPERATURE)
        adv, ret = compute_gae(rewards, values, dones, last_values, active_mask)
        batch = flatten_ppo_batch((features, valid, base_score, chosen, logp, values, ret, adv, active_mask))
        sample_n = batch[0].shape[0]
        mb = int(MINIBATCH_SIZE)
        loss = jnp.asarray(0.0, dtype=jnp.float32)
        aux = (jnp.asarray(0.0, dtype=jnp.float32), jnp.asarray(0.0, dtype=jnp.float32), jnp.asarray(0.0, dtype=jnp.float32))
        for _epoch in range(int(UPDATE_EPOCHS)):
            for start in range(0, sample_n, mb):
                mini = minibatch_slice(batch, start, mb)
                params, opt_state, loss, aux = ppo_update(params, opt_state, mini)

        if (update + 1) % EVAL_EVERY_UPDATES == 0 or update == 0:
            mean_reward = np.asarray(rewards).sum(axis=0).mean(axis=0)
            loss_f = float(loss)
            pol_f, val_f, ent_f = [float(x) for x in aux]
            rec = {
                'update': int(start_update) + int(update) + 1,
                'loss': loss_f,
                'policy_loss': pol_f,
                'value_loss': val_f,
                'entropy': ent_f,
                'mean_player_reward': mean_reward.tolist(),
                'active_sample_rate': float(np.asarray(active_mask).mean()),
                'done_rate': float(np.asarray(dones).mean()),
            }
            history.append(rec)
            pbar.set_postfix(loss=f'{loss_f:.4f}', entropy=f'{ent_f:.3f}', reward=np.round(mean_reward, 3).tolist())
            print('update', int(start_update) + int(update) + 1, 'loss', loss_f, 'policy', pol_f, 'value', val_f, 'entropy', ent_f, 'mean_player_reward', mean_reward, 'active_rate', float(np.asarray(active_mask).mean()), 'done_rate', float(np.asarray(dones).mean()))

        if (update + 1) % SAVE_EVERY_UPDATES == 0:
            global_update = int(start_update) + int(update) + 1
            ckpt_path = OUT_DIR / f'ppo_policy_update_{global_update:04d}.npz'
            save_policy_params(params, ckpt_path)
            save_policy_params(params, OUT_DIR / 'ppo_policy_latest.npz')
            hist_path = OUT_DIR / 'train_history.json'
            hist_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
            print('checkpoint saved at update', global_update)

        if (update + 1) % RESET_EVERY_UPDATES == 0:
            state = reset_state
            reset_state = make_initial_jax_batch(seed_start=seed + 200000 + int(start_update + update + 1) * 31, batch_size=num_envs)

    globals()['TRAIN_HISTORY'] = history
    save_policy_params(params, OUT_DIR / 'ppo_policy_latest.npz')
    (OUT_DIR / 'train_history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
    return params

print('JAX PPO V24-like candidate port ready. Run train_ppo(seed=0) after validation passes.')


In [ ]:
# === SAVE / LOAD TRAINED PARAMS ===
# Sau khi train: params = train_ppo(seed=0)
# Luu ra npz de co the gan vao submission runtime sau nay.


def save_policy_params(params, path=OUT_DIR / 'ppo_policy_params.npz'):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {name: np.asarray(getattr(params, name)) for name in params._fields}
    np.savez_compressed(path, **arrays)
    print('saved', path)
    return path


def load_policy_params(path=OUT_DIR / 'ppo_policy_params.npz'):
    data = np.load(Path(path))
    return PolicyParams(**{name: jnp.asarray(data[name]) for name in PolicyParams._fields})

# Example:
# params = train_ppo(seed=0)
# save_policy_params(params)


In [ ]:
# === OPTIONAL TRAIN RUN ===
# Mac dinh AUTO_START_TRAIN=False nen Run all chi validate, khong train dai.
# Khi muon train that, sua AUTO_START_TRAIN=True trong cell config roi Run all / run cell nay.

TRAINED_PARAMS = None
if AUTO_START_TRAIN:
    if AUTO_RUN_VALIDATION and not VALIDATION_OK:
        raise RuntimeError('Validation did not pass, training is blocked.')

    if RESUME_FROM_POLICY:
        print('Resume from policy checkpoint:', RESUME_POLICY_PATH)
        base_params = load_policy_params(RESUME_POLICY_PATH)
        remaining_updates = max(0, int(TOTAL_PPO_UPDATES) - int(RESUME_START_UPDATE))
        print('RESUME_START_UPDATE', RESUME_START_UPDATE)
        print('remaining_updates', remaining_updates)
        TRAINED_PARAMS = train_ppo(seed=0, updates=remaining_updates, initial_params=base_params, start_update=RESUME_START_UPDATE)
    else:
        TRAINED_PARAMS = train_ppo(seed=0)

    save_policy_params(TRAINED_PARAMS)
else:
    print('AUTO_START_TRAIN=False, skipped long PPO train.')
    print('To train: set AUTO_START_TRAIN=True in config, then run all again.')
    print('Resume: set RESUME_FROM_POLICY=True and RESUME_POLICY_PATH to your .npz checkpoint.')
    print('Or run manually: params = train_ppo(seed=0); save_policy_params(params)')


## Ket luan thiet ke

Ban nay train PPO tren JAX simulator voi candidate generator port theo V24 o muc thuc dung.
No chua phai port 100% `submission_v24_good.py`, nhung da gan hon heuristic nho ban dau: source shortlist, target shortlist, safe drain, focus-fire, regroup/defense candidates.
